# Starters, and pull TMDB dataset

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree

In [ ]:
!pip install kagglehub[pandas-datasets]

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "TMDB_movie_dataset_v11.csv"

# Load the latest version
tmdb = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "asaniczka/tmdb-movies-dataset-2023-930k-movies",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", tmdb.head())

<ipython-input-94-61cebd4c1d67>:8: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  tmdb = kagglehub.load_dataset(


100%|██████████| 516M/516M [00:12<00:00, 42.8MB/s]


First 5 records:        id            title  vote_average  vote_count    status release_date  \
0   27205        Inception         8.364       34495  Released   2010-07-15   
1  157336     Interstellar         8.417       32571  Released   2014-11-05   
2     155  The Dark Knight         8.512       30619  Released   2008-07-16   
3   19995           Avatar         7.573       29815  Released   2009-12-15   
4   24428     The Avengers         7.710       29166  Released   2012-04-25   

      revenue  runtime  adult                     backdrop_path  ...  \
0   825532764      148  False  /8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg  ...   
1   701729206      169  False  /pbrkL804c8yAv3zBZR4QPEafpAR.jpg  ...   
2  1004558444      152  False  /nMKdUUepR0i5zn0y1T4CsSB5chy.jpg  ...   
3  2923706026      162  False  /vL5LR6WdxWPjLPFRLe133jXWsh5.jpg  ...   
4  1518815515      143  False  /9BBTo63ANSmhC4e6r62OJFuK2GL.jpg  ...   

    original_title                                           overview  \
0 

# Useful tools

In [ ]:
tmdb[tmdb['imdb_id'].str.contains("tt9214832", case=False, na=False)][['title_year','overview']]


,title_year,overview
2698,Emma. (2020),"In 1800s England, a well-meaning but selfish y..."


In [ ]:
tmdb[tmdb['title_year'].str.contains("straight story", case=False, na=False)][['id','title_year','imdb_id','overview']]


,id,title_year,imdb_id,overview
3062,404,The Straight Story (1999),tt0166896,"A retired farmer and widower in his 70s, Alvin..."
45597,48128,Straight Story (2006),tt0976216,"This film is about an upside down world, where..."
1058282,898395,A Straight Story (2021),NaN,"In A Straight Story, a train journey between P..."


# TMDB: One-time processing

In [ ]:
tmdb['title'] = tmdb['title'].str.replace('&', 'and', regex=False)


In [ ]:
tmdb['title (year)'] = tmdb['title'].astype(str) + ' (' + tmdb['release_date'].astype(str).str[:4] + ')'


In [ ]:
tmdb = tmdb.rename(columns={'title (year)': 'title_year'})


In [ ]:
tmdb_select = tmdb[["id", "imdb_id", "original_language", "genres", "production_countries", "keywords", "title_year"]]

In [ ]:
tmdb_select = tmdb_select.rename(columns={'id': 't_id'})

# Load PC, merge to TMDB_select_nodup

In [ ]:

# Replace 'your_file.xlsx' with the actual path to your Excel file
pc = pd.read_excel('e_popchart_movie_data.xlsx')

# Now you can work with the data in the 'df' DataFrame
print(pc.head())


                title                 title_year  year  genre       theme  \
0         City Lights         City Lights (1931)  1931  Drama  Love Story   
1  As Good as It Gets  As Good as It Gets (1997)  1997  Drama  Love Story   
2              Amélie              Amélie (2001)  2001  Drama  Love Story   
3        Pretty Woman        Pretty Woman (1990)  1990  Drama  Love Story   
4           Manhattan           Manhattan (1979)  1979  Drama  Love Story   

    theme2 theme3  
0  Rom-Com    NaN  
1  Rom-Com    NaN  
2  Rom-Com    NaN  
3  Rom-Com    NaN  
4  Rom-Com    NaN  


In [ ]:
pc['title_year_orig'] = pc['title'].astype(str) + ' (' + pc['year'].astype(str) + ')'

In [ ]:
merged_df = pd.merge(pc, tmdb_select_nodup, on='title_year', how='left')

print(merged_df.head())

                title                 title_year  year  genre       theme  \
0         City Lights         City Lights (1931)  1931  Drama  Love Story   
1  As Good as It Gets  As Good as It Gets (1997)  1997  Drama  Love Story   
2              Amélie              Amélie (2001)  2001  Drama  Love Story   
3        Pretty Woman        Pretty Woman (1990)  1990  Drama  Love Story   
4           Manhattan           Manhattan (1979)  1979  Drama  Love Story   

    theme2 theme3  t_id    imdb_id original_language                  genres  \
0  Rom-Com    NaN   901  tt0021749                en  Comedy, Romance, Drama   
1  Rom-Com    NaN  2898  tt0119822                en  Drama, Comedy, Romance   
2  Rom-Com    NaN   194  tt0211915                fr         Comedy, Romance   
3  Rom-Com    NaN   114  tt0100405                en         Comedy, Romance   
4  Rom-Com    NaN   696  tt0079522                en  Comedy, Drama, Romance   

       production_countries                             

In [ ]:
nan_id_count = merged_df['t_id'].isna().sum()
print(f"Number of NaN values in 'id' column: {nan_id_count}")

Number of NaN values in 'id' column: 0


In [ ]:
merged_df['imdb_id'].isna().sum()

0

In [ ]:
merged_df.to_excel('merged_df.xlsx', index=False)

# Special letter check

In [ ]:
# prompt: given a list of values, how do I identify values with special letters? (letters not in the english alphabet)

import re

def identify_special_letters(values):
  """
  Identifies values in a list that contain special characters (non-English alphabet letters).

  Args:
    values: A list of strings.

  Returns:
    A list of strings containing values with special characters.
  """
  special_values = []
  for value in values:
    # Check if the value is a string before applying regex
    if isinstance(value, str):
      if re.search(r'[^\x00-\x7F]+', value):  # Matches any character outside the ASCII range
          special_values.append(value)
  return special_values



In [ ]:
# Example usage (assuming 'df' is your DataFrame and 'title' is your column)
# Replace with your actual column name
titles_with_special_chars = identify_special_letters(pc['name'].tolist())

titles_with_special_chars

['Say Anything…',
 'When Harry Met Sally…',
 'There’s Something About Mary',
 'What’s Up, Doc?',
 'Adam’s Rib',
 'Who’s Afraid of Virginia Woolf?',
 'Y Tu Mamá También',
 'Lola Montès',
 'Ivan’s Childhood',
 'L’Avventura',
 'Sophie’s Choice',
 'One Flew Over the Cuckoo’s Nest',
 'Lorenzo’s Oil',
 'I’m Not There',
 'What’s Love Got to Do with It',
 'The King’s Speech',
 'Schindler’s List',
 'Boys Don’t Cry',
 'All the President’s Men',
 'Romy and Michele’s High School Reunion',
 'White Men Can’t Jump',
 'A Hard Day’s Night',
 'Don’t Be a Menace to South Central While Drinking Your Juice in the Hood',
 'The Band’s Visit',
 'Ferris Bueller’s Day Off',
 'Sullivan’s Travels',
 'National Lampoon’s Vacation',
 'Who’s Harry Crumb?',
 'Monsieur Hulot’s Holiday',
 'Ulzana’s Raid',
 'Hang ’em High',
 'Heaven’s Gate',
 'Carlito’s Way',
 'Miller’s Crossing',
 'Ocean’s 11',
 'Le Samouraï',
 'Léon: The Professional',
 'Caché',
 'You’re Next',
 'Don’t Breathe',
 'The Double Life of Véronique',
 'It’s 

In [ ]:
sp_letter_titles = ['Y Tu Mamá También','Lola Montès','Le Samouraï','Léon: The Professional','Caché','The Double Life of Véronique','Nausicaä of the Valley of the Wind']

In [ ]:
sp_letter_titles

['Y Tu Mamá También',
 'Lola Montès',
 'Le Samouraï',
 'Léon: The Professional',
 'Caché',
 'The Double Life of Véronique',
 'Nausicaä of the Valley of the Wind']

In [ ]:
sp_letter_titles_en = ['Y Tu Mama Tambien',
 'Lola Montes',
 'Le Samourai',
 'Leon: The Professional',
 'Cache',
 'The Double Life of Veronique',
 'Nausicaa of the Valley of the Wind']

In [ ]:
for i in range(len(sp_letter_titles)):
  pc['name'] = pc['name'].str.replace(sp_letter_titles_en[i], sp_letter_titles[i], regex=False)

In [ ]:
tmdb[tmdb['title'].str.contains(sp_letter_titles[6], case=False, na=False)]['title']

,title
1408,Nausicaä of the Valley of the Wind


# Remove dups from tmdb

In [ ]:

title_year_counts = tmdb['title_year'].value_counts()


title_years_to_check = title_year_counts[title_year_counts >= 2].index
tys = []

for title_year in merged_df['title_year']:
    if title_year in title_years_to_check:

        tys.append(title_year)

NameError: name 'merged_df' is not defined

In [ ]:
tyss = sorted(list(set(tys)))
len(tyss)

42

In [ ]:
ids_del = [678094, 1070315, 969829, 343693, 883188, 472349, 1310146, 1284516, 339098, 822740, 641185, 869857, 996894, 1065063, 501007, 688994, 1430208, 875501, 227977, 645443, 441705, 385938, 473993, 1189676, 1164820, 1095006, 1003632, 1435787, 1187746, 1103859, 1391422, 314836, 1220707, 536708, 1088159, 612176, 364520, 1214190, 1329489, 633548, 1359588, 444686, 263514, 1365861, 526667, 240763, 671975, 1095268, 1295439]
len(ids_del)

49

In [ ]:
#Checked that t_ids to delete are unique
#tmdb_select[tmdb_select['t_id'].isin(ids_del)].groupby('t_id').size()

In [ ]:
tmdb_select_nodup = tmdb_select[~tmdb_select['t_id'].isin(ids_del)]

In [ ]:
tmdb_select.shape
tmdb_select_nodup.shape

(1188053, 7)

(1188004, 7)

# Work on TMDB Genres and Keywords

In [ ]:
t_genres = ['Comedy',  'Drama',  'Romance',  'Crime',  'Action',  'Music',  'Western',  'War',  'Thriller',  'Adventure',  'Animation',  'History',  'Mystery',  'Documentary',  'Science Fiction',  'Family',  'Horror',  'Fantasy']

In [ ]:
len(t_genres)

18

In [ ]:
import pandas as pd

In [ ]:
kws = pd.read_excel('keywords.xlsx')

In [ ]:

all_values_set = set()

for col in kws.columns:

  for value in kws[col].dropna():
    all_values_set.add(value)

all_values_set


{'nosferatu',
 'water skiing',
 'pest control',
 'emperor',
 'bodily dismemberment',
 'arabian',
 'bible quote',
 'based on magazine',
 'holiday season',
 'generation z',
 'love at first sight',
 'flea',
 'religious art',
 'courtroom drama',
 'gentrification',
 'cheyenne',
 'thirty something',
 'graduation',
 'giving away money',
 'wild west',
 'swinging 60s',
 'east india company',
 'mutant',
 'montana',
 'shopping',
 '5th century',
 'inventor',
 'drug humor',
 'cheerleader',
 'charming man',
 'botched robbery',
 'quest for identity',
 'pep rally',
 'pulitzer prize',
 'human sacrifice',
 'geologist',
 'sex party',
 'stapler',
 'chemist',
 'gang violence',
 'teenage mortality',
 '12th century',
 'cape cod',
 'testosterone',
 'approved school ',
 'fishing boat',
 'napoleon bonaparte',
 'stockholm syndrome',
 'water slide',
 'vaudeville',
 'dreary',
 'boy and dog',
 'terror cell',
 'existentialism',
 'taj mahal',
 'cattle',
 'heavy metal',
 'sword',
 'externally controlled action',
 'oil

In [ ]:
len(all_values_set)

7567

In [ ]:
merged_df = pd.read_excel('merged_df.xlsx')

In [ ]:
merged_df.head()

,title,title_year,year,genre,theme,theme2,theme3,title_year_orig,t_id,imdb_id,original_language,genres,production_countries,keywords
0,City Lights,City Lights (1931),1931,Drama,Love Story,Rom-Com,NaN,City Lights (1931),901,tt0021749,en,"Comedy, Romance, Drama",United States of America,"blindness and impaired vision, eye operation, ..."
1,As Good as It Gets,As Good as It Gets (1997),1997,Drama,Love Story,Rom-Com,NaN,As Good as It Gets (1997),2898,tt0119822,en,"Drama, Comedy, Romance",United States of America,"friendship, waitress, single parent, restauran..."
2,Amélie,Amélie (2001),2001,Drama,Love Story,Rom-Com,NaN,Amélie (2001),194,tt0211915,fr,"Comedy, Romance","France, Germany","daughter, paris, france, love triangle, photog..."
3,Pretty Woman,Pretty Woman (1990),1990,Drama,Love Story,Rom-Com,NaN,Pretty Woman (1990),114,tt0100405,en,"Comedy, Romance",United States of America,"hotel, friendship, prostitute, sports car, cap..."
4,Manhattan,Manhattan (1979),1979,Drama,Love Story,Rom-Com,NaN,Manhattan (1979),696,tt0079522,en,"Comedy, Drama, Romance",United States of America,"adultery, lolita, new york city, based on nove..."


In [ ]:
from collections import defaultdict

keyword_counts = defaultdict(int)
for keywords in merged_df['keywords'].dropna():
    for keyword in keywords.split(', '):
        keyword_counts[keyword] += 1

keywords_atleast = [keyword for keyword, count in keyword_counts.items() if count >= 3]


In [ ]:
len(keywords_atleast)

1915

In [ ]:
keywords_atleast[:5]

['blindness and impaired vision',
 'operation',
 "love of one's life",
 'suicide attempt',
 'tramp']

In [ ]:


row_count = 0
for index, row in merged_df.iterrows():
    if isinstance(row['keywords'], str):
      keywords_in_row = row['keywords'].split(', ')
      for keyword in keywords_in_row:
          if keyword in keywords_atleast:
              row_count += 1
              break

print(f"Number of rows with at least one keyword from 'keywords_atleast_25': {row_count}")


Number of rows with at least one keyword from 'keywords_atleast_25': 1229


# iMDB Linkage

In [ ]:
i_basics = pd.read_csv('title.basics.tsv', sep='\t')

i_basics.head()


<ipython-input-60-3d890bbd7294>:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  i_basics = pd.read_csv('title.basics.tsv', sep='\t')


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [ ]:
i_basics.shape

(11493130, 9)

In [ ]:
pc_i_basics = i_basics[i_basics['tconst'].isin(merged_df['imdb_id'])]
pc_i_basics = pc_i_basics[['tconst', 'primaryTitle', 'originalTitle', 'startYear', 'genres']]

In [ ]:
pc_i_basics.head()

,tconst,primaryTitle,originalTitle,startYear,genres
10181,tt0010323,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,1920,"Horror,Mystery,Thriller"
12169,tt0012349,The Kid,The Kid,1921,"Comedy,Drama,Family"
13240,tt0013442,Nosferatu: A Symphony of Horror,"Nosferatu, eine Symphonie des Grauens",1922,"Fantasy,Horror"
14199,tt0014429,Safety Last!,Safety Last!,1923,"Action,Comedy,Thriller"
15081,tt0015324,Sherlock Jr.,Sherlock Jr.,1924,"Action,Comedy,Romance"


In [ ]:
i_ratings = pd.read_csv('title.ratings.tsv', sep='\t')

i_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2138
1,tt0000002,5.5,289
2,tt0000003,6.4,2171
3,tt0000004,5.3,185
4,tt0000005,6.2,2905


In [ ]:
i_ratings.shape

(1540609, 3)

In [ ]:
merged_imdb = pd.merge(pc_i_basics, i_ratings, on='tconst', how='left')

In [ ]:
merged_imdb.to_excel('merged_imdb.xlsx', index=False)

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
merged_imdb['numVotes'].describe()


,numVotes
count,1256.000
mean,218452.707
std,347026.144
min,2441.000
25%,31457.000
50%,91064.500
75%,243031.500
max,3013434.000


In [ ]:
merged_df = merged_df.rename(columns={'genre': 'pc_genre', 'genres': 'tmdb_genres', 't_id': 'tmdb_id', 'keywords': 'tmdb_themes'})
merged_imdb = merged_imdb.rename(columns={'primaryTitle': 'imdb_primaryTitle', 'originalTitle': 'imdb_originalTitle', 'startYear': 'imdb_startYear', 'genres': 'imdb_genres', 'averageRating': 'imdb_rating'})

NameError: name 'merged_imdb' is not defined

In [ ]:
merged_imdb = merged_imdb.drop(columns=['numVotes', 'imdb_primaryTitle', 'imdb_originalTitle', 'imdb_startYear'])

In [ ]:
pti_df = pd.merge(merged_df, merged_imdb, left_on='imdb_id', right_on='tconst', how='left')

In [ ]:
pti_df.to_excel('pti_df.xlsx', index=False)

In [ ]:
pti_df.columns

Index(['title', 'title_year', 'year', 'pc_genre', 'theme', 'theme2', 'theme3',
       'tmdb_id', 'imdb_id', 'original_language', 'tmdb_genres',
       'production_countries', 'tmdb_themes', 'tconst', 'imdb_genres',
       'imdb_rating'],
      dtype='object')

In [ ]:
pti_df.drop(columns=['tconst'], inplace=True)

In [ ]:
pti_df_short = pti_df[['imdb_id', 'title', 'year', 'title_year', 'original_language', 'production_countries', 'imdb_rating']]

In [ ]:
pti_df_short.head()

,imdb_id,title,year,title_year,original_language,production_countries,imdb_rating
0,tt0021749,City Lights,1931,City Lights (1931),en,United States of America,8.5
1,tt0119822,As Good as It Gets,1997,As Good as It Gets (1997),en,United States of America,7.7
2,tt0211915,Amélie,2001,Amélie (2001),fr,"France, Germany",8.3
3,tt0100405,Pretty Woman,1990,Pretty Woman (1990),en,United States of America,7.1
4,tt0079522,Manhattan,1979,Manhattan (1979),en,United States of America,7.8


# Merge with OMDB data

In [ ]:
omdb = pd.read_excel('omdb_with_rt.xlsx')

In [ ]:
omdb = omdb.rename(columns={'Genre': 'og'})


Continuing maximal data work at the bottom

In [ ]:
omdb_genre = omdb[['imdbID', 'og']]

In [ ]:
genres = pd.merge(df_3, omdb_genre, left_on='imdb_id', right_on='imdbID', how='left')

In [ ]:
genres = genres[['title_year', 'imdb_id', 'pc_genre', 'imdb_genres', 'tmdb_genres', 'og']]

In [ ]:
genres = genres.rename(columns={'pc_genre': 'pg', 'imdb_genres': 'ig', 'tmdb_genres': 'tg'})

# Combining PC + TMDB + IMDB + OMDB genres

In [ ]:
genres['og'] = genres['og'].str.replace(', ', ',', regex=False)
genres['ig'] = genres['ig'].str.replace(', ', ',', regex=False)
genres['pg'] = genres['pg'].str.replace(', ', ',', regex=False)
genres['tg'] = genres['tg'].str.replace(', ', ',', regex=False)

In [ ]:
genres['og'] = genres['og'].str.lower()
genres['ig'] = genres['ig'].str.lower()
genres['pg'] = genres['pg'].str.lower()
genres['tg'] = genres['tg'].str.lower()

In [ ]:
genres['pg'] = genres['pg'].str.replace('based on a true story', 'true_story', regex=False)
genres['tg'] = genres['tg'].str.replace('science fiction', 'sci-fi', regex=False)
genres['og'] = genres['og'].str.replace('short,', '', regex=False)

In [ ]:
cols = ['og', 'pg', 'ig', 'tg']
genres['combined'] = genres[cols].apply(lambda row: ','.join(row.dropna().astype(str)), axis=1)

In [ ]:
genres['combined'] = genres['combined'].str.split(',')
genres['combined'] = genres['combined'].apply(lambda x: list(set(x)))

In [ ]:
genres.to_excel('genres.xlsx', index=False)

# Link RottenTomatoes

In [ ]:

file_path = "rotten_tomatoes_movies.csv"

# Load the latest version
rt_raw = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", rt_raw.head())

<ipython-input-18-ddc501763ad5>:4: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  rt_raw = kagglehub.load_dataset(


100%|██████████| 6.76M/6.76M [00:00<00:00, 39.5MB/s]

Extracting zip of rotten_tomatoes_movies.csv...


First 5 records:                      id                title  audienceScore  tomatoMeter  \
0    space-zombie-bingo  Space Zombie Bingo!           50.0          NaN   
1       the_green_grass      The Green Grass            NaN          NaN   
2             love_lies           Love, Lies           43.0          NaN   
3  the_sore_losers_1997          Sore Losers           60.0          NaN   
4  dinosaur_island_2002      Dinosaur Island           70.0          NaN   

  rating ratingContents releaseDateTheaters releaseDateStreaming  \
0    NaN            NaN                 NaN           2018-08-25   
1    NaN            NaN                 NaN           2020-02-11   
2    NaN            NaN                 NaN                  NaN   
3    NaN            NaN                 NaN           2020-10-23   
4    NaN            NaN                 NaN           2017-03-27   

   runtimeMinutes                          genre originalLanguage  \
0            75.0         Comedy, Horror, Sci-fi

In [ ]:
rt = rt_raw[rt_raw['tomatoMeter'].notna()].copy()


In [ ]:
rt.shape

(33877, 16)

In [ ]:
rt.to_excel('rt.xlsx', index=False)

In [ ]:
rt.head()

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
5,adrift_2018,Adrift,65.000,69.000,PG-13,"['Injury Images', 'Brief Drug Use', 'Thematic ...",2018-06-01,2018-08-21,120.000,"Adventure, Drama, Romance",English,Baltasar Kormákur,"Aaron Kandell,Jordan Kandell,David Branson Smith",$31.4M,STX Films,NaN
9,1035316-born_to_kill,Born to Kill,74.000,83.000,NaN,NaN,1947-04-30,2016-05-23,92.000,"Crime, Drama",English,Robert Wise,"Eve Greene,Richard Macaulay",NaN,NaN,NaN
12,margarita_happy_hour,Margarita Happy Hour,NaN,76.000,NaN,NaN,2002-03-22,NaN,98.000,Drama,English,Ilya Chaiken,Ilya Chaiken,$11.5K,Passport Pictures,Surround
13,leap_of_faith_2019,Leap of Faith: William Friedkin on The Exorcist,86.000,93.000,NaN,NaN,NaN,2020-11-19,104.000,"Documentary, Mystery & thriller",English,Alexandre O. Philippe,Alexandre O. Philippe,NaN,NaN,NaN
17,1221483-paa,Paa,67.000,50.000,NaN,NaN,2009-12-04,NaN,133.000,Drama,Hindi,R. Balki,R. Balki,$199.2K,Big Pictures,NaN


In [ ]:
rt['release_year'] = rt['releaseDateTheaters']
rt['release_year'] = rt['release_year'].fillna(rt['releaseDateStreaming'].astype(str).str[:4])
rt['release_year'] = rt['release_year'].astype(str).str[:4]

In [ ]:
rt['title_year'] = rt['title'].astype(str) + ' (' + rt['release_year'].astype(str) + ')'

In [ ]:
rt.head()

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix,release_year,title_year
5,adrift_2018,Adrift,65.000,69.000,PG-13,"['Injury Images', 'Brief Drug Use', 'Thematic ...",2018-06-01,2018-08-21,120.000,"Adventure, Drama, Romance",English,Baltasar Kormákur,"Aaron Kandell,Jordan Kandell,David Branson Smith",$31.4M,STX Films,NaN,2018,Adrift (2018)
9,1035316-born_to_kill,Born to Kill,74.000,83.000,NaN,NaN,1947-04-30,2016-05-23,92.000,"Crime, Drama",English,Robert Wise,"Eve Greene,Richard Macaulay",NaN,NaN,NaN,1947,Born to Kill (1947)
12,margarita_happy_hour,Margarita Happy Hour,NaN,76.000,NaN,NaN,2002-03-22,NaN,98.000,Drama,English,Ilya Chaiken,Ilya Chaiken,$11.5K,Passport Pictures,Surround,2002,Margarita Happy Hour (2002)
13,leap_of_faith_2019,Leap of Faith: William Friedkin on The Exorcist,86.000,93.000,NaN,NaN,NaN,2020-11-19,104.000,"Documentary, Mystery & thriller",English,Alexandre O. Philippe,Alexandre O. Philippe,NaN,NaN,NaN,2020,Leap of Faith: William Friedkin on The Exorcis...
17,1221483-paa,Paa,67.000,50.000,NaN,NaN,2009-12-04,NaN,133.000,Drama,Hindi,R. Balki,R. Balki,$199.2K,Big Pictures,NaN,2009,Paa (2009)


In [ ]:
rt_small = rt[['title_year', 'tomatoMeter']]

In [ ]:
df_3['r_year'] = df_3['title_year'].str[-5:-1].astype(int)

In [ ]:
df_3['title_yf'] = df_3['title_year'].str[:-7] + ' (' + (df_3['r_year'] + 1).astype(str) + ')'

In [ ]:
df_3['title_yb'] = df_3['title_year'].str[:-7] + ' (' + (df_3['r_year'] - 1).astype(str) + ')'

In [ ]:
df_3.head()

,title,title_year,pc_year,pc_genre,theme,theme2,theme3,title_year_orig,tmdb_id,imdb_id,...,tmdb_genres,production_countries,tmdb_themes,imdb_genres,imdb_rating,ty_p,ty_f,r_year,title_yf,title_yb
0,City Lights,City Lights (1931),1931,Drama,Love Story,Rom-Com,NaN,City Lights (1931),901,tt0021749,...,"Comedy, Romance, Drama",United States of America,"blindness and impaired vision, eye operation, ...","Comedy,Drama,Romance",8.500,1930,1932,1931,City Lights (1932),City Lights (1930)
1,As Good as It Gets,As Good as It Gets (1997),1997,Drama,Love Story,Rom-Com,NaN,As Good as It Gets (1997),2898,tt0119822,...,"Drama, Comedy, Romance",United States of America,"friendship, waitress, single parent, restauran...","Comedy,Drama,Romance",7.700,1996,1998,1997,As Good as It Gets (1998),As Good as It Gets (1996)
2,Amélie,Amélie (2001),2001,Drama,Love Story,Rom-Com,NaN,Amélie (2001),194,tt0211915,...,"Comedy, Romance","France, Germany","daughter, paris, france, love triangle, photog...","Comedy,Romance",8.300,2000,2002,2001,Amélie (2002),Amélie (2000)
3,Pretty Woman,Pretty Woman (1990),1990,Drama,Love Story,Rom-Com,NaN,Pretty Woman (1990),114,tt0100405,...,"Comedy, Romance",United States of America,"hotel, friendship, prostitute, sports car, cap...","Comedy,Romance",7.100,1989,1991,1990,Pretty Woman (1991),Pretty Woman (1989)
4,Manhattan,Manhattan (1979),1979,Drama,Love Story,Rom-Com,NaN,Manhattan (1979),696,tt0079522,...,"Comedy, Drama, Romance",United States of America,"adultery, lolita, new york city, based on nove...","Comedy,Drama,Romance",7.800,1978,1980,1979,Manhattan (1980),Manhattan (1978)


In [ ]:
df_4 = pd.merge(df_3, rt_small, on='title_year', how='left')

# Merge on 'title_yf' if 'tomatoMeter' is NaN
df_4_yf = pd.merge(df_4[df_4['tomatoMeter'].isna()], rt_small, left_on='title_yf', right_on='title_year', how='left', suffixes=('', '_yf'))
df_4_yf = df_4_yf.drop(columns=['title_year_yf']) # Remove duplicated column
df_4 = pd.concat([df_4[~df_4['tomatoMeter'].isna()], df_4_yf], ignore_index=True) # Changed to pd.concat

# Merge on 'title_yb' if 'tomatoMeter' is still NaN
df_4_yb = pd.merge(df_4[df_4['tomatoMeter'].isna()], rt_small, left_on='title_yb', right_on='title_year', how='left', suffixes=('', '_yb'))
df_4_yb = df_4_yb.drop(columns=['title_year_yb']) # Remove duplicated column
df_4 = pd.concat([df_4[~df_4['tomatoMeter'].isna()], df_4_yb], ignore_index=True) # Changed to pd.concat

In [ ]:
df_4.to_excel('df_4.xlsx', index=False)

In [ ]:
df_4.to_excel('df_4.xlsx', index=False)

In [ ]:
rt[rt['title'].str.contains("fireflies", case=False, na=False)][['tomatoMeter', 'title_year', 'director', 'releaseDateTheaters', 'releaseDateStreaming']]


,tomatoMeter,title_year,director,releaseDateTheaters,releaseDateStreaming
45070,23.000,Fireflies in the Garden (2011),Dennis Lee,2011-10-14,2012-02-07
63308,100.000,Grave of the Fireflies (2017),Isao Takahata,NaN,2017-03-11
92574,100.000,The Fireflies Are Gone (nan),Sébastien Pilote,NaN,NaN


# Metascore binding

In [ ]:
# Set the path to the file you'd like to load
file_path = "imdb-movies-dataset.csv"

# Load the latest version
imdb_to_ms = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "amanbarthwal/imdb-movies-data",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", imdb_to_ms.head())

<ipython-input-23-3cd197e05131>:5: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  imdb_to_ms = kagglehub.load_dataset(


100%|██████████| 8.02M/8.02M [00:00<00:00, 47.0MB/s]

Extracting zip of imdb-movies-dataset.csv...


First 5 records:                                               Poster  \
0  https://m.media-amazon.com/images/M/MV5BYWRkZj...   
1  https://m.media-amazon.com/images/M/MV5BZGI4NT...   
2  https://m.media-amazon.com/images/M/MV5BZjIyOT...   
3  https://m.media-amazon.com/images/M/MV5BMjA5Zj...   
4  https://m.media-amazon.com/images/M/MV5BNTk1MT...   

                               Title    Year Certificate  Duration (min)  \
0                    The Idea of You  2023.0           R           115.0   
1  Kingdom of the Planet of the Apes  2023.0       PG-13           145.0   
2                          Unfrosted  2023.0       PG-13            97.0   
3                       The Fall Guy  2023.0       PG-13           126.0   
4                        Challengers  2023.0           R           131.0   

                        Genre  Rating  Metascore           Director  \
0      Comedy, Drama, Romance     6.4       67.0  Michael Showalter   
1   Action, Adventure, Sci-Fi     7.3       66.

In [ ]:
imdb_to_ms_f = imdb_to_ms[imdb_to_ms['Metascore'].notna()].copy()


In [ ]:
imdb_to_ms_f.shape

(7555, 15)

In [ ]:
imdb_to_ms_f.to_excel('imdb_to_ms_f.xlsx', index=False)

# imdb_id list

In [ ]:
iid1 = ['tt0010323', 'tt0012349', 'tt0013442', 'tt0014429', 'tt0015324', 'tt0015648', 'tt0015864', 'tt0016039', 'tt0017136', 'tt0017925', 'tt0018037', 'tt0018455', 'tt0019254', 'tt0020629', 'tt0021749', 'tt0021814', 'tt0021884', 'tt0022100', 'tt0022286', 'tt0022827', 'tt0022913', 'tt0023027', 'tt0023238', 'tt0023427', 'tt0023969', 'tt0024034', 'tt0024184', 'tt0024188', 'tt0024210', 'tt0024216', 'tt0024894', 'tt0025316', 'tt0025878', 'tt0026029', 'tt0026138', 'tt0026174', 'tt0027125', 'tt0027977', 'tt0028445', 'tt0029442', 'tt0029583', 'tt0029843', 'tt0029947', 'tt0030643', 'tt0031381', 'tt0031679', 'tt0031725', 'tt0031762', 'tt0031885', 'tt0031971', 'tt0032138', 'tt0032455', 'tt0032551', 'tt0032553', 'tt0032599', 'tt0032762', 'tt0032904', 'tt0032910', 'tt0032976', 'tt0033152', 'tt0033467', 'tt0033704', 'tt0033804', 'tt0033870', 'tt0034240', 'tt0034398', 'tt0034583', 'tt0034587', 'tt0035015', 'tt0035169', 'tt0035211', 'tt0036027', 'tt0036112', 'tt0036506', 'tt0036613', 'tt0036695', 'tt0036775', 'tt0036868', 'tt0037008', 'tt0037075', 'tt0037558', 'tt0037674', 'tt0038348', 'tt0038355', 'tt0038559', 'tt0038650', 'tt0038733', 'tt0038787', 'tt0039192', 'tt0039204', 'tt0039220', 'tt0039305', 'tt0039689', 'tt0040522', 'tt0040536', 'tt0040724', 'tt0040725', 'tt0040897', 'tt0041090', 'tt0041546', 'tt0041719', 'tt0041786', 'tt0041931', 'tt0041959', 'tt0042192', 'tt0042208', 'tt0042276', 'tt0042332', 'tt0042393', 'tt0042436', 'tt0042546', 'tt0042593', 'tt0042804', 'tt0042876', 'tt0043014', 'tt0043265', 'tt0043274', 'tt0043278', 'tt0043338', 'tt0043456', 'tt0043924', 'tt0043961', 'tt0044079', 'tt0044081', 'tt0044121', 'tt0044207', 'tt0044419', 'tt0044706', 'tt0044953', 'tt0045061', 'tt0045070', 'tt0045152', 'tt0045537', 'tt0045546', 'tt0045555', 'tt0045793', 'tt0045848', 'tt0045920', 'tt0046187', 'tt0046250', 'tt0046268', 'tt0046303', 'tt0046359', 'tt0046438', 'tt0046487', 'tt0046511', 'tt0046521', 'tt0046534', 'tt0046754', 'tt0046876', 'tt0046911', 'tt0046912', 'tt0047034', 'tt0047296', 'tt0047396', 'tt0047437', 'tt0047445', 'tt0047478', 'tt0047528', 'tt0047573', 'tt0047577', 'tt0047677', 'tt0048021', 'tt0048028', 'tt0048261', 'tt0048281', 'tt0048308', 'tt0048424', 'tt0048452', 'tt0048473', 'tt0048491', 'tt0048545', 'tt0048605', 'tt0048728', 'tt0048977', 'tt0049010', 'tt0049223', 'tt0049261', 'tt0049366', 'tt0049406', 'tt0049408', 'tt0049470', 'tt0049646', 'tt0049730', 'tt0049778', 'tt0049782', 'tt0049902', 'tt0049966', 'tt0050083', 'tt0050212', 'tt0050280', 'tt0050407', 'tt0050419', 'tt0050530', 'tt0050539', 'tt0050585', 'tt0050613', 'tt0050706', 'tt0050766', 'tt0050825', 'tt0050976', 'tt0050986', 'tt0051036', 'tt0051201', 'tt0051337', 'tt0051365', 'tt0051378', 'tt0051380', 'tt0051411', 'tt0051418', 'tt0051459', 'tt0051554', 'tt0051622', 'tt0051745', 'tt0051994', 'tt0052218', 'tt0052311', 'tt0052357', 'tt0052561', 'tt0052618', 'tt0052646', 'tt0052893', 'tt0052918', 'tt0052948', 'tt0053085', 'tt0053121', 'tt0053125', 'tt0053146', 'tt0053198', 'tt0053221', 'tt0053285', 'tt0053291', 'tt0053459', 'tt0053472', 'tt0053604', 'tt0053619', 'tt0053779', 'tt0054047', 'tt0054067', 'tt0054167', 'tt0054215', 'tt0054248', 'tt0054331', 'tt0054443', 'tt0054632', 'tt0054953', 'tt0054997', 'tt0055018', 'tt0055031', 'tt0055032', 'tt0055184', 'tt0055254', 'tt0055601', 'tt0055614', 'tt0055630', 'tt0055830', 'tt0055928', 'tt0056111', 'tt0056172', 'tt0056210', 'tt0056217', 'tt0056218', 'tt0056406', 'tt0056412', 'tt0056592', 'tt0056687', 'tt0056869', 'tt0056875', 'tt0057012', 'tt0057076', 'tt0057091', 'tt0057115', 'tt0057129', 'tt0057197', 'tt0057345', 'tt0057565', 'tt0057603', 'tt0057693', 'tt0057869', 'tt0058138', 'tt0058150', 'tt0058182', 'tt0058279', 'tt0058331', 'tt0058385', 'tt0058450', 'tt0058530', 'tt0058586', 'tt0058604', 'tt0058700', 'tt0058898', 'tt0058946', 'tt0059017', 'tt0059084', 'tt0059578', 'tt0059601', 'tt0059646', 'tt0059742', 'tt0060107', 'tt0060138', 'tt0060176', 'tt0060196', 'tt0060315', 'tt0060390', 'tt0060665', 'tt0060782', 'tt0060841', 'tt0060955', 'tt0061065', 'tt0061184', 'tt0061395', 'tt0061418', 'tt0061429', 'tt0061512', 'tt0061537', 'tt0061578', 'tt0061709', 'tt0061722', 'tt0061747', 'tt0061809', 'tt0061811', 'tt0061852', 'tt0062138', 'tt0062229', 'tt0062262', 'tt0062622', 'tt0062711', 'tt0062765', 'tt0062952', 'tt0062994', 'tt0063032', 'tt0063152', 'tt0063350', 'tt0063374', 'tt0063442', 'tt0063462', 'tt0063501', 'tt0063522', 'tt0063678', 'tt0063823', 'tt0064040', 'tt0064068', 'tt0064115', 'tt0064116', 'tt0064276', 'tt0064451', 'tt0064665', 'tt0065126', 'tt0065143', 'tt0065163', 'tt0065214', 'tt0065234', 'tt0065377', 'tt0065466', 'tt0065531', 'tt0065571', 'tt0065988', 'tt0066026', 'tt0066206', 'tt0066214', 'tt0066380', 'tt0066434', 'tt0066849', 'tt0066921', 'tt0066999', 'tt0067128', 'tt0067185', 'tt0067328', 'tt0067411', 'tt0067525', 'tt0067656', 'tt0067756', 'tt0067800', 'tt0067805', 'tt0067866', 'tt0067893', 'tt0067959', 'tt0067961', 'tt0067992', 'tt0068182', 'tt0068327', 'tt0068361', 'tt0068473', 'tt0068646', 'tt0068675', 'tt0068699', 'tt0068767', 'tt0069089', 'tt0069113', 'tt0069293', 'tt0069341', 'tt0069436', 'tt0069495', 'tt0069704', 'tt0069738', 'tt0069762', 'tt0069945', 'tt0069995', 'tt0070034', 'tt0070040', 'tt0070047', 'tt0070334', 'tt0070359', 'tt0070379', 'tt0070460', 'tt0070510', 'tt0070518', 'tt0070531', 'tt0070544', 'tt0070622', 'tt0070666', 'tt0070698', 'tt0070707', 'tt0070723', 'tt0070735', 'tt0070800', 'tt0070842', 'tt0070849', 'tt0070903', 'tt0070909', 'tt0070917', 'tt0071222', 'tt0071230', 'tt0071249', 'tt0071315', 'tt0071360', 'tt0071562', 'tt0071615', 'tt0071853', 'tt0071970', 'tt0072271', 'tt0072308', 'tt0072417', 'tt0072431', 'tt0072684', 'tt0072730', 'tt0072856', 'tt0072890', 'tt0072962', 'tt0073195', 'tt0073312', 'tt0073341', 'tt0073440', 'tt0073486', 'tt0073580', 'tt0073582', 'tt0073629', 'tt0073692', 'tt0073802', 'tt0073812', 'tt0074119', 'tt0074156', 'tt0074174', 'tt0074256', 'tt0074285', 'tt0074486', 'tt0074593', 'tt0074695', 'tt0074812', 'tt0074851', 'tt0074958', 'tt0075005', 'tt0075029', 'tt0075148', 'tt0075213', 'tt0075314', 'tt0075686', 'tt0075860', 'tt0076054', 'tt0076263', 'tt0076504', 'tt0076683', 'tt0076723', 'tt0076729', 'tt0076759', 'tt0076786', 'tt0076929', 'tt0077138', 'tt0077402', 'tt0077405', 'tt0077416', 'tt0077559', 'tt0077631', 'tt0077651', 'tt0077663', 'tt0077681', 'tt0077745', 'tt0077914', 'tt0077928', 'tt0077975', 'tt0078087', 'tt0078243', 'tt0078346', 'tt0078446', 'tt0078748', 'tt0078754', 'tt0078767', 'tt0078788', 'tt0078841', 'tt0078902', 'tt0078908', 'tt0078935', 'tt0079367', 'tt0079470', 'tt0079501', 'tt0079522', 'tt0079714', 'tt0079944', 'tt0080040', 'tt0080057', 'tt0080120', 'tt0080130', 'tt0080339', 'tt0080354', 'tt0080360', 'tt0080365', 'tt0080455', 'tt0080487', 'tt0080516', 'tt0080678', 'tt0080684', 'tt0080745', 'tt0080749', 'tt0080761', 'tt0080855', 'tt0081114', 'tt0081283', 'tt0081383', 'tt0081398', 'tt0081455', 'tt0081505', 'tt0081562', 'tt0081633', 'tt0082010', 'tt0082089', 'tt0082096', 'tt0082118', 'tt0082136', 'tt0082158', 'tt0082186', 'tt0082198', 'tt0082242', 'tt0082269', 'tt0082307', 'tt0082340', 'tt0082348', 'tt0082427', 'tt0082533', 'tt0082694', 'tt0082748', 'tt0082782', 'tt0082869', 'tt0082971', 'tt0082979', 'tt0083336', 'tt0083511', 'tt0083630', 'tt0083658', 'tt0083767', 'tt0083791', 'tt0083866', 'tt0083907', 'tt0083929', 'tt0083944', 'tt0083946', 'tt0083987', 'tt0084237', 'tt0084516', 'tt0084707', 'tt0084726', 'tt0084787', 'tt0084805', 'tt0084827', 'tt0084855', 'tt0085333', 'tt0085334', 'tt0085382', 'tt0085407', 'tt0085794', 'tt0085811', 'tt0085852', 'tt0085859', 'tt0085862', 'tt0085995', 'tt0086200', 'tt0086250', 'tt0086336', 'tt0086429', 'tt0086465', 'tt0086541', 'tt0086617', 'tt0086637', 'tt0086837', 'tt0086879', 'tt0086960', 'tt0086979', 'tt0087004', 'tt0087015', 'tt0087050', 'tt0087182', 'tt0087262', 'tt0087298', 'tt0087332', 'tt0087363', 'tt0087538', 'tt0087544', 'tt0087553', 'tt0087781', 'tt0087799', 'tt0087800', 'tt0087843', 'tt0087884', 'tt0087909', 'tt0087957', 'tt0087995', 'tt0088000', 'tt0088117', 'tt0088128', 'tt0088161', 'tt0088247', 'tt0088258', 'tt0088323', 'tt0088763', 'tt0088846', 'tt0088847', 'tt0088933', 'tt0088944', 'tt0088993', 'tt0089013', 'tt0089092', 'tt0089175', 'tt0089200', 'tt0089218', 'tt0089374', 'tt0089457', 'tt0089469', 'tt0089504', 'tt0089791', 'tt0089881', 'tt0089885', 'tt0089893', 'tt0089907', 'tt0090021', 'tt0090094', 'tt0090142', 'tt0090190', 'tt0090555', 'tt0090563', 'tt0090605', 'tt0090728', 'tt0090756', 'tt0090863', 'tt0090887', 'tt0090917', 'tt0090967', 'tt0091042', 'tt0091059', 'tt0091064', 'tt0091203', 'tt0091217', 'tt0091223', 'tt0091251', 'tt0091369', 'tt0091499', 'tt0091630', 'tt0091635', 'tt0091738', 'tt0091763', 'tt0091790', 'tt0091954', 'tt0092005', 'tt0092099', 'tt0092513', 'tt0092610', 'tt0092675', 'tt0092699', 'tt0092843', 'tt0092991', 'tt0093010', 'tt0093058', 'tt0093075', 'tt0093177', 'tt0093185', 'tt0093409', 'tt0093560', 'tt0093565', 'tt0093593', 'tt0093605', 'tt0093640', 'tt0093677', 'tt0093748', 'tt0093773', 'tt0093779', 'tt0093822', 'tt0093870', 'tt0093894', 'tt0094012', 'tt0094025', 'tt0094027', 'tt0094226', 'tt0094336', 'tt0094625', 'tt0094631', 'tt0094721', 'tt0094737', 'tt0094812', 'tt0094862', 'tt0094898', 'tt0094924', 'tt0094961', 'tt0095016', 'tt0095031', 'tt0095037', 'tt0095082', 'tt0095159', 'tt0095270', 'tt0095327', 'tt0095631', 'tt0095647', 'tt0095705', 'tt0095765', 'tt0095925', 'tt0095953', 'tt0096163', 'tt0096251', 'tt0096256', 'tt0096283', 'tt0096409', 'tt0096438', 'tt0096446', 'tt0096764', 'tt0096895', 'tt0096913', 'tt0096969', 'tt0097108', 'tt0097123', 'tt0097165', 'tt0097202', 'tt0097216', 'tt0097236', 'tt0097351', 'tt0097441', 'tt0097493', 'tt0097576', 'tt0097659', 'tt0097757', 'tt0097815', 'tt0097937', 'tt0098061', 'tt0098067', 'tt0098084', 'tt0098143', 'tt0098206', 'tt0098258', 'tt0098354', 'tt0098635', 'tt0098645', 'tt0098724', 'tt0099077', 'tt0099334', 'tt0099348', 'tt0099365', 'tt0099487', 'tt0099685', 'tt0099740', 'tt0099763', 'tt0099785', 'tt0100142', 'tt0100150', 'tt0100157', 'tt0100263', 'tt0100405', 'tt0100449', 'tt0100802', 'tt0100814', 'tt0100934', 'tt0100944', 'tt0101410', 'tt0101414', 'tt0101507', 'tt0101700', 'tt0101765', 'tt0101862', 'tt0102057', 'tt0102059', 'tt0102250', 'tt0102587', 'tt0102592', 'tt0102685', 'tt0102926', 'tt0103064', 'tt0103074', 'tt0103285', 'tt0103639', 'tt0103772', 'tt0103873', 'tt0103893', 'tt0103919', 'tt0104036', 'tt0104257', 'tt0104348', 'tt0104558', 'tt0104684', 'tt0104691', 'tt0104694', 'tt0104756', 'tt0104797', 'tt0104868', 'tt0105104', 'tt0105151', 'tt0105236', 'tt0105690', 'tt0105695', 'tt0105812', 'tt0106308', 'tt0106489', 'tt0106519', 'tt0106611', 'tt0106673', 'tt0106677', 'tt0106977', 'tt0107048', 'tt0107206', 'tt0107207', 'tt0107290', 'tt0107473', 'tt0107614', 'tt0107688', 'tt0107818', 'tt0107943', 'tt0108002', 'tt0108037', 'tt0108052', 'tt0108122', 'tt0108148', 'tt0108160', 'tt0108358', 'tt0108394', 'tt0108399', 'tt0108551', 'tt0109040', 'tt0109348', 'tt0109424', 'tt0109445', 'tt0109506', 'tt0109592', 'tt0109686', 'tt0109688', 'tt0109707', 'tt0109759', 'tt0109830', 'tt0109831', 'tt0110005', 'tt0110308', 'tt0110357', 'tt0110413', 'tt0110912', 'tt0110932', 'tt0111112', 'tt0111161', 'tt0111257', 'tt0111282', 'tt0111495', 'tt0111503', 'tt0111512', 'tt0111579', 'tt0111686', 'tt0111742', 'tt0112288', 'tt0112384', 'tt0112431', 'tt0112442', 'tt0112471', 'tt0112508', 'tt0112573', 'tt0112641', 'tt0112682', 'tt0112697', 'tt0112817', 'tt0112818', 'tt0112864', 'tt0113118', 'tt0113247', 'tt0113277', 'tt0113540', 'tt0113568', 'tt0113627', 'tt0114272', 'tt0114323', 'tt0114367', 'tt0114369', 'tt0114558', 'tt0114694', 'tt0114709', 'tt0114746', 'tt0114814', 'tt0115678', 'tt0115685', 'tt0115734', 'tt0115736', 'tt0116126', 'tt0116209', 'tt0116282', 'tt0116367', 'tt0116483', 'tt0116594', 'tt0116629', 'tt0116683', 'tt0116695', 'tt0116743', 'tt0116778', 'tt0116905', 'tt0116922', 'tt0117008', 'tt0117060', 'tt0117318', 'tt0117438', 'tt0117500', 'tt0117571', 'tt0117589', 'tt0117631', 'tt0117666', 'tt0117951', 'tt0117998', 'tt0118111', 'tt0118632', 'tt0118655', 'tt0118715', 'tt0118749', 'tt0118799', 'tt0118884', 'tt0118929', 'tt0118971', 'tt0119081', 'tt0119116', 'tt0119167', 'tt0119174', 'tt0119177', 'tt0119217', 'tt0119229', 'tt0119256', 'tt0119345', 'tt0119349', 'tt0119396', 'tt0119488', 'tt0119654', 'tt0119675', 'tt0119698', 'tt0119711', 'tt0119822', 'tt0120032', 'tt0120188', 'tt0120201', 'tt0120255', 'tt0120263', 'tt0120265', 'tt0120324', 'tt0120338', 'tt0120363', 'tt0120382', 'tt0120586', 'tt0120601', 'tt0120611', 'tt0120623', 'tt0120630', 'tt0120660', 'tt0120663', 'tt0120669', 'tt0120679', 'tt0120684', 'tt0120689', 'tt0120735', 'tt0120737', 'tt0120762', 'tt0120780', 'tt0120812', 'tt0120815', 'tt0120863', 'tt0120885', 'tt0120903', 'tt0120907', 'tt0121164', 'tt0122690', 'tt0125664', 'tt0126029', 'tt0126886', 'tt0127536', 'tt0128442', 'tt0128445']

In [ ]:
iid2 = ['tt0129167', 'tt0129332', 'tt0129387', 'tt0133093', 'tt0134119', 'tt0137523', 'tt0138097', 'tt0138704', 'tt0139654', 'tt0140352', 'tt0144084', 'tt0145487', 'tt0146838', 'tt0146882', 'tt0147612', 'tt0149261', 'tt0151568', 'tt0151804', 'tt0154420', 'tt0158371', 'tt0158983', 'tt0159097', 'tt0160916', 'tt0162222', 'tt0162346', 'tt0163988', 'tt0165078', 'tt0165854', 'tt0166396', 'tt0166924', 'tt0167190', 'tt0167260', 'tt0167261', 'tt0167404', 'tt0169547', 'tt0171804', 'tt0172493', 'tt0172495', 'tt0175880', 'tt0177789', 'tt0180093', 'tt0181689', 'tt0181865', 'tt0183659', 'tt0185014', 'tt0185125', 'tt0185937', 'tt0189998', 'tt0190332', 'tt0195685', 'tt0195714', 'tt0196229', 'tt0198781', 'tt0203009', 'tt0203119', 'tt0203166', 'tt0203230', 'tt0206634', 'tt0208092', 'tt0209144', 'tt0210070', 'tt0210945', 'tt0211915', 'tt0215129', 'tt0216216', 'tt0217869', 'tt0218839', 'tt0230600', 'tt0238380', 'tt0240772', 'tt0241303', 'tt0241527', 'tt0243017', 'tt0243655', 'tt0244316', 'tt0245429', 'tt0245574', 'tt0246578', 'tt0247196', 'tt0247425', 'tt0248667', 'tt0248845', 'tt0249462', 'tt0253474', 'tt0256009', 'tt0257360', 'tt0258000', 'tt0258463', 'tt0259711', 'tt0264464', 'tt0264616', 'tt0265343', 'tt0265666', 'tt0266308', 'tt0266543', 'tt0266697', 'tt0268126', 'tt0272338', 'tt0274558', 'tt0274812', 'tt0276751', 'tt0277434', 'tt0280609', 'tt0282698', 'tt0286106', 'tt0287467', 'tt0289043', 'tt0296572', 'tt0297884', 'tt0298130', 'tt0299658', 'tt0299977', 'tt0300214', 'tt0302886', 'tt0304141', 'tt0305206', 'tt0311113', 'tt0312004', 'tt0316188', 'tt0317248', 'tt0317705', 'tt0319061', 'tt0319343', 'tt0325980', 'tt0327056', 'tt0327597', 'tt0329575', 'tt0332379', 'tt0333766', 'tt0335266', 'tt0337563', 'tt0338013', 'tt0338751', 'tt0340377', 'tt0340855', 'tt0343818', 'tt0347149', 'tt0349825', 'tt0350258', 'tt0356721', 'tt0357413', 'tt0358273', 'tt0363163', 'tt0363547', 'tt0363589', 'tt0363771', 'tt0364385', 'tt0364569', 'tt0365748', 'tt0366548', 'tt0367089', 'tt0368794', 'tt0368913', 'tt0369339', 'tt0369702', 'tt0371746', 'tt0373074', 'tt0374900', 'tt0375063', 'tt0375912', 'tt0376541', 'tt0376968', 'tt0377092', 'tt0379725', 'tt0379786', 'tt0381061', 'tt0381668', 'tt0381681', 'tt0382932', 'tt0383028', 'tt0385004', 'tt0386117', 'tt0387564', 'tt0387808', 'tt0387898', 'tt0388795', 'tt0390022', 'tt0390221', 'tt0390384', 'tt0395169', 'tt0399146', 'tt0399201', 'tt0401383', 'tt0401792', 'tt0402399', 'tt0404203', 'tt0405094', 'tt0405159', 'tt0405296', 'tt0405422', 'tt0407887', 'tt0408790', 'tt0410097', 'tt0412019', 'tt0415306', 'tt0416320', 'tt0416449', 'tt0420087', 'tt0420223', 'tt0425112', 'tt0427944', 'tt0432283', 'tt0434409', 'tt0435625', 'tt0435761', 'tt0436697', 'tt0437857', 'tt0439815', 'tt0443272', 'tt0443453', 'tt0443680', 'tt0443706', 'tt0448115', 'tt0448134', 'tt0449059', 'tt0450278', 'tt0451279', 'tt0452039', 'tt0452594', 'tt0453562', 'tt0454876', 'tt0455590', 'tt0455760', 'tt0455944', 'tt0456149', 'tt0457430', 'tt0457510', 'tt0460829', 'tt0461770', 'tt0462499', 'tt0463854', 'tt0464141', 'tt0468489', 'tt0468492', 'tt0468569', 'tt0469494', 'tt0470752', 'tt0475276', 'tt0476735', 'tt0477348', 'tt0478304', 'tt0478311', 'tt0479884', 'tt0480249', 'tt0482571', 'tt0482606', 'tt0486655', 'tt0493464', 'tt0497465', 'tt0499549', 'tt0758758', 'tt0765429', 'tt0765443', 'tt0780504', 'tt0780521', 'tt0780536', 'tt0796366', 'tt0800039', 'tt0808417', 'tt0808506', 'tt0816692', 'tt0816711', 'tt0829482', 'tt0838283', 'tt0841046', 'tt0848228', 'tt0851578', 'tt0861739', 'tt0862856', 'tt0878804', 'tt0887912', 'tt0892769', 'tt0898367', 'tt0907657', 'tt0910970', 'tt0926084', 'tt0929632', 'tt0936501', 'tt0959337', 'tt0964517', 'tt0166896', 'tt0985694', 'tt0993846', 'tt1010048', 'tt1017460', 'tt1019452', 'tt1022603', 'tt1024648', 'tt1029234', 'tt10293406', 'tt1032846', 'tt1032856', 'tt1034415', 'tt10365998', 'tt1038988', 'tt1045658', 'tt1049413', 'tt10514222', 'tt1051906', 'tt1060277', 'tt1065073', 'tt1068646', 'tt1077258', 'tt10954984', 'tt1100089', 'tt11084432', 'tt1109624', 'tt1119646', 'tt11252440', 'tt1125849', 'tt11271038', 'tt1127180', 'tt1136608', 'tt1136617', 'tt1139797', 'tt11426232', 'tt11525644', 'tt1156398', 'tt11564570', 'tt1160419', 'tt1172994', 'tt1174732', 'tt1179904', 'tt1179933', 'tt11813216', 'tt1182345', 'tt11866324', 'tt1186830', 'tt1187064', 'tt1213641', 'tt1220719', 'tt1226681', 'tt1226774', 'tt1232829', 'tt1250777', 'tt1259521', 'tt1270798', 'tt1276104', 'tt1278340', 'tt1282140', 'tt13172796', 'tt1320253', 'tt1321510', 'tt13238346', 'tt13270424', 'tt1334260', 'tt1340800', 'tt1341167', 'tt1343727', 'tt1375666', 'tt1392170', 'tt1392190', 'tt1392214', 'tt1396484', 'tt1399683', 'tt14230388', 'tt14230458', 'tt1431045', 'tt1436045', 'tt1441326', 'tt1441395', 'tt14444726', 'tt1448754', 'tt14509110', 'tt1454468', 'tt1457767', 'tt1462758', 'tt1467304', 'tt1470827', 'tt1478338', 'tt1478964', 'tt1483013', 'tt1489887', 'tt1490017', 'tt14905650', 'tt1504320', 'tt15090124', 'tt1517268', 'tt1517451', 'tt15239678', 'tt1534085', 'tt15398776', 'tt1541874', 'tt1542344', 'tt1549572', 'tt15893750', 'tt1591095', 'tt1594562', 'tt1596343', 'tt1605783', 'tt1610996', 'tt1612774', 'tt1620935', 'tt16300962', 'tt1631867', 'tt1659337', 'tt1663202', 'tt1663662', 'tt1675434', 'tt1687901', 'tt17009710', 'tt1706620', 'tt1772341', 'tt1798709', 'tt1825683', 'tt1853614', 'tt1853728', 'tt1853739', 'tt1856101', 'tt1878870', 'tt1899353', 'tt1950186', 'tt19770238', 'tt1977895', 'tt1979320', 'tt1979376', 'tt2015381', 'tt2017038', 'tt20215968', 'tt2024544', 'tt2051879', 'tt2084989', 'tt2096673', 'tt2105044', 'tt21064584', 'tt2106476', 'tt21192142', 'tt2119532', 'tt2209418', 'tt2245084', 'tt2250912', 'tt2267998', 'tt2272350', 'tt2278871', 'tt2294629', 'tt2309021', 'tt2321405', 'tt2321549', 'tt2326554', 'tt23289160', 'tt2349144', 'tt2359024', 'tt2380307', 'tt2388715', 'tt2397535', 'tt2400463', 'tt2406566', 'tt2420756', 'tt2428170', 'tt2488496', 'tt2494362', 'tt2535470', 'tt2543164', 'tt2554274', 'tt2562232', 'tt2576852', 'tt2582782', 'tt2582802', 'tt2597242', 'tt2716382', 'tt2781516', 'tt2788732', 'tt2793490', 'tt2798920', 'tt2802144', 'tt2818178', 'tt2820852', 'tt2837574', 'tt2866360', 'tt2911666', 'tt2935510', 'tt2948372', 'tt3040964', 'tt3086442', 'tt3104988', 'tt3235888', 'tt3263904', 'tt3286052', 'tt3289956', 'tt3315342', 'tt3322940', 'tt3387648', 'tt3395184', 'tt3397884', 'tt3416532', 'tt3416742', 'tt3464902', 'tt3498820', 'tt3501632', 'tt3520418', 'tt3521126', 'tt3521164', 'tt3564472', 'tt3581652', 'tt3606756', 'tt3608930', 'tt3659388', 'tt3691740', 'tt3726704', 'tt3748528', 'tt3783958', 'tt3882000', 'tt3885422', 'tt3890160', 'tt3960412', 'tt3967856', 'tt3976144', 'tt3986820', 'tt4034228', 'tt4052882', 'tt4062536', 'tt4086032', 'tt4105970', 'tt4116284', 'tt4154664', 'tt4154756', 'tt4154796', 'tt4160708', 'tt4196450', 'tt4218572', 'tt4263482', 'tt4266076', 'tt4291590', 'tt4302938', 'tt4468740', 'tt4530422', 'tt4547056', 'tt4575576', 'tt4595882', 'tt4633694', 'tt4669986', 'tt4680182', 'tt4686844', 'tt4695012', 'tt4827558', 'tt4846340', 'tt4925292', 'tt4935334', 'tt4954522', 'tt4975722', 'tt4978710', 'tt5013056', 'tt5022702', 'tt5027774', 'tt5052448', 'tt5073642', 'tt5083738', 'tt5104604', 'tt5109784', 'tt5215952', 'tt5308322', 'tt5311514', 'tt5437928', 'tt5439796', 'tt5442430', 'tt5462602', 'tt5516328', 'tt5566790', 'tt5580036', 'tt5580390', 'tt5592248', 'tt5657856', 'tt5688932', 'tt5700672', 'tt5715874', 'tt5726616', 'tt5727208', 'tt5742374', 'tt5776858', 'tt5848272', 'tt5867314', 'tt5918982', 'tt5990474', 'tt5999530', 'tt6155172', 'tt6217306', 'tt6217608', 'tt6243140', 'tt6265828', 'tt6292852', 'tt6491178', 'tt6499752', 'tt6587046', 'tt6644200', 'tt6710474', 'tt6803046', 'tt6857112', 'tt6900448', 'tt6908274', 'tt6998518', 'tt7014006', 'tt7137380', 'tt7349662', 'tt7589524', 'tt7653254', 'tt7784604', 'tt7914416', 'tt7946422', 'tt8508734', 'tt8579674', 'tt8772262', 'tt9005184', 'tt9016974', 'tt9048804', 'tt9072352', 'tt9214832', 'tt9231040', 'tt9243804', 'tt9362722', 'tt9484998', 'tt9731534', 'tt9738716', 'tt9770150', 'tt9784798']

# OMDB Pull

In [ ]:
len(iid1)

989

In [ ]:
import requests
import pandas as pd

def get_movie_data(imdb_id, api_key):
  """
  Retrieves movie data as a dictionary using the OMDb API.

  Args:
    imdb_id: The IMDb ID of the movie.
    api_key: Your OMDb API key.

  Returns:
    The movie data as a dictionary, or None if an error occurs.
  """
  url = f"http://www.omdbapi.com/?i={imdb_id}&plot=full&apikey={api_key}"

  try:
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for bad status codes

    data = response.json()
    if data["Response"] == "True":
      return data  # Return the JSON data as a dictionary
    else:
      print(f"Error for {imdb_id}: {data.get('Error', 'Unknown error')}")
      return None
  except requests.exceptions.RequestException as e:
    print(f"An error occurred for {imdb_id}: {e}")
    return None

# Your OMDb API key
api_key = "d35bdbbf"  # Replace with your actual API key

# Create an empty list to store the movie data
all_movie_data = []

# Iterate through the IMDb IDs and retrieve the movie data
for imdb_id in iid1:
  movie_data = get_movie_data(imdb_id, api_key)
  if movie_data:
    all_movie_data.append(movie_data)

# Create a DataFrame from the list of movie data
movies_df = pd.DataFrame(all_movie_data)

# Now you have a single DataFrame 'movies_df' containing data for all 990 movies
print(movies_df.head())

                             Title  Year      Rated     Released Runtime  \
0      The Cabinet of Dr. Caligari  1920  Not Rated  27 Feb 1920  67 min   
1                          The Kid  1921     Passed  06 Feb 1921  68 min   
2  Nosferatu: A Symphony of Horror  1922  Not Rated  18 May 1922  94 min   
3                     Safety Last!  1923  Not Rated  01 Apr 1923  74 min   
4                     Sherlock Jr.  1924     Passed  11 May 1924  45 min   

                       Genre                      Director  \
0  Horror, Mystery, Thriller                  Robert Wiene   
1      Comedy, Drama, Family               Charles Chaplin   
2            Fantasy, Horror                   F.W. Murnau   
3   Action, Comedy, Thriller  Fred C. Newmeyer, Sam Taylor   
4    Action, Comedy, Romance                 Buster Keaton   

                                              Writer  \
0                          Carl Mayer, Hans Janowitz   
1                                    Charles Chaplin   
2 

In [ ]:
movies_df.to_excel('movies_df.xlsx')

In [ ]:
# Your OMDb API key
api_key = "f2dacbe5"  # Replace with your actual API key

# Create an empty list to store the movie data
all_movie_data2 = []

# Iterate through the IMDb IDs and retrieve the movie data
for imdb_id in iid2:
  movie_data = get_movie_data(imdb_id, api_key)
  if movie_data:
    all_movie_data2.append(movie_data)

# Create a DataFrame from the list of movie data
movies_df2 = pd.DataFrame(all_movie_data2)

# Now you have a single DataFrame 'movies_df' containing data for all 990 movies
print(movies_df2.head())

                          Title  Year Rated     Released  Runtime  \
0                The Iron Giant  1999    PG  06 Aug 1999   86 min   
1                      Ravenous  1999     R  19 Mar 1999  101 min   
2  There's Something About Mary  1998     R  15 Jul 1998  119 min   
3                    The Matrix  1999     R  31 Mar 1999  136 min   
4       The Talented Mr. Ripley  1999     R  25 Dec 1999  139 min   

                          Genre                         Director  \
0  Animation, Action, Adventure                        Brad Bird   
1      Adventure, Drama, Horror                     Antonia Bird   
2               Comedy, Romance   Bobby Farrelly, Peter Farrelly   
3                Action, Sci-Fi  Lana Wachowski, Lilly Wachowski   
4        Crime, Drama, Thriller                Anthony Minghella   

                                       Writer  \
0        Tim McCanlies, Brad Bird, Ted Hughes   
1                                 Ted Griffin   
2  Ed Decter, John J. Strauss

In [ ]:

combined_df = pd.concat([movies_df, movies_df2], ignore_index=True)
print(combined_df.head())


                             Title  Year      Rated     Released Runtime  \
0      The Cabinet of Dr. Caligari  1920  Not Rated  27 Feb 1920  67 min   
1                          The Kid  1921     Passed  06 Feb 1921  68 min   
2  Nosferatu: A Symphony of Horror  1922  Not Rated  18 May 1922  94 min   
3                     Safety Last!  1923  Not Rated  01 Apr 1923  74 min   
4                     Sherlock Jr.  1924     Passed  11 May 1924  45 min   

                       Genre                      Director  \
0  Horror, Mystery, Thriller                  Robert Wiene   
1      Comedy, Drama, Family               Charles Chaplin   
2            Fantasy, Horror                   F.W. Murnau   
3   Action, Comedy, Thriller  Fred C. Newmeyer, Sam Taylor   
4    Action, Comedy, Romance                 Buster Keaton   

                                              Writer  \
0                          Carl Mayer, Hans Janowitz   
1                                    Charles Chaplin   
2 

In [ ]:
combined_df.to_excel('omdb.xlsx')

# Stanford movie reviews

In [ ]:
!tar -xzvf acl.tar.gz


Streaming output truncated to the last 5000 lines.
aclImdb/train/unsup/44983_0.txt
aclImdb/train/unsup/44982_0.txt
aclImdb/train/unsup/44981_0.txt
aclImdb/train/unsup/44980_0.txt
aclImdb/train/unsup/44979_0.txt
aclImdb/train/unsup/44978_0.txt
aclImdb/train/unsup/44977_0.txt
aclImdb/train/unsup/44976_0.txt
aclImdb/train/unsup/44975_0.txt
aclImdb/train/unsup/44974_0.txt
aclImdb/train/unsup/44973_0.txt
aclImdb/train/unsup/44972_0.txt
aclImdb/train/unsup/44971_0.txt
aclImdb/train/unsup/44970_0.txt
aclImdb/train/unsup/44969_0.txt
aclImdb/train/unsup/44968_0.txt
aclImdb/train/unsup/44967_0.txt
aclImdb/train/unsup/44966_0.txt
aclImdb/train/unsup/44965_0.txt
aclImdb/train/unsup/44964_0.txt
aclImdb/train/unsup/44963_0.txt
aclImdb/train/unsup/44962_0.txt
aclImdb/train/unsup/44961_0.txt
aclImdb/train/unsup/44960_0.txt
aclImdb/train/unsup/44959_0.txt
aclImdb/train/unsup/44958_0.txt
aclImdb/train/unsup/44957_0.txt
aclImdb/train/unsup/44956_0.txt
aclImdb/train/unsup/44955_0.txt
aclImdb/train/unsup/4

In [ ]:
import pandas as pd
import os

def create_dataframe_from_txt_files(directory, urls):
    '''Creates dataframe from a .txt file'''
    data = {'review': [], 'iid': []}
    for i, filename in enumerate(os.listdir(directory)):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory, filename)
            try:
                with open(filepath, 'r', encoding='utf-8') as file:
                    content = file.read()
                    data['review'].append(content)
                    data['iid'].append(urls[i])
            except Exception as e:
                print(f"Error reading file {filename}: {e}")
                data['review'].append(None)
                data['iid'].append(None)

    df = pd.DataFrame(data)
    return df


                                              review        iid
0  Well, I set out with a few friends to see this...  tt0406816
1  A low-rent, cheaply made police thriller that'...  tt0406816
2  The beginning of the film is promising. When J...  tt0406816
3  I saw Bogard when it was released in the 70s. ...  tt0406816
4  I have been a fan of the Carpenters for a long...  tt0406816


In [ ]:
import pandas as pd



In [ ]:

file_path = 'aclImdb/test/urls_neg.txt'
dftest_neg = pd.read_csv(file_path, header=None, names=['url'])

urltest_neg = []

for index, row in dftest_neg.iterrows():
    line = row['url']
    match = re.search(r'/title/(tt\d+)/usercomments', line)
    if match:
        urltest_neg.append(match.group(1))

print(len(urltest_neg))

directory_path = 'aclImdb/test/neg'
revtest_neg = create_dataframe_from_txt_files(directory_path, urltest_neg)
print(revtest_neg.head())

none_count = revtest_neg[revtest_neg['iid'].isnull()].shape[0]
print(f"Number of 'None' values in 'iid' column: {none_count}")


12500
                                              review        iid
0  Well, I set out with a few friends to see this...  tt0406816
1  A low-rent, cheaply made police thriller that'...  tt0406816
2  The beginning of the film is promising. When J...  tt0406816
3  I saw Bogard when it was released in the 70s. ...  tt0406816
4  I have been a fan of the Carpenters for a long...  tt0406816
Number of 'None' values in 'iid' column: 0


In [ ]:

file_path = 'aclImdb/test/urls_pos.txt'
dftest_pos = pd.read_csv(file_path, header=None, names=['url'])

urltest_pos = []

for index, row in dftest_pos.iterrows():
    line = row['url']
    match = re.search(r'/title/(tt\d+)/usercomments', line)
    if match:
        urltest_pos.append(match.group(1))

print(len(urltest_pos))

directory_path = 'aclImdb/test/pos'
revtest_pos = create_dataframe_from_txt_files(directory_path, urltest_pos)
print(revtest_pos.head())

none_count = revtest_pos[revtest_pos['iid'].isnull()].shape[0]
print(f"Number of 'None' values in 'iid' column: {none_count}")


12500
                                              review        iid
0  Thank goodness not all Dutch people are that r...  tt0406816
1  When I saw this movie at age 6, it was in the ...  tt0406816
2  Frank Horrigan (Clint Eastwood) is a secret se...  tt0406816
3  To solve a challenging problem, you need to st...  tt0406816
4  Even an old cynical DOCTOR WHO fan like myself...  tt0406816
Number of 'None' values in 'iid' column: 0


In [ ]:

file_path = 'aclImdb/train/urls_pos.txt'
dftrain_pos = pd.read_csv(file_path, header=None, names=['url'])

urltrain_pos = []

for index, row in dftrain_pos.iterrows():
    line = row['url']
    match = re.search(r'/title/(tt\d+)/usercomments', line)
    if match:
        urltrain_pos.append(match.group(1))

print(len(urltrain_pos))

directory_path = 'aclImdb/train/pos'
revtrain_pos = create_dataframe_from_txt_files(directory_path, urltrain_pos)
print(revtrain_pos.head())

none_count = revtrain_pos[revtrain_pos['iid'].isnull()].shape[0]
print(f"Number of 'None' values in 'iid' column: {none_count}")

12500
                                              review        iid
0  After 'Aakrosh' , this was the second film for...  tt0453418
1  Critically, people say that Antz is better. An...  tt0453418
2  Ah, clichés, clichés, clichés; They're a main ...  tt0453418
3  If you enjoy Cleese & all the British 'Pythone...  tt0064354
4  ***SPOILERS*** Seething with hatred and reveng...  tt0064354
Number of 'None' values in 'iid' column: 0


In [ ]:
file_path = 'aclImdb/train/urls_neg.txt'
dftrain_neg = pd.read_csv(file_path, header=None, names=['url'])

urltrain_neg = []

for index, row in dftrain_neg.iterrows():
    line = row['url']
    match = re.search(r'/title/(tt\d+)/usercomments', line)
    if match:
        urltrain_neg.append(match.group(1))

print(len(urltrain_neg))

directory_path = 'aclImdb/train/neg'
revtrain_neg = create_dataframe_from_txt_files(directory_path, urltrain_neg)
print(revtrain_neg.head())

none_count = revtrain_neg[revtrain_neg['iid'].isnull()].shape[0]
print(f"Number of 'None' values in 'iid' column: {none_count}")

12500
                                              review        iid
0  i was disappointed. the film was a bit predict...  tt0064354
1  Well I have to admit this was one of my favori...  tt0100680
2  The Three Stooges in a feature length western ...  tt0100680
3  Florence Chadwick was actually the far more ac...  tt0100680
4  I first heard of Unisol 2 when I drove past a ...  tt0047200
Number of 'None' values in 'iid' column: 0


In [ ]:
file_path = 'aclImdb/train/urls_unsup.txt'
dftrain_unsup = pd.read_csv(file_path, header=None, names=['url'])

urltrain_unsup = []

for index, row in dftrain_unsup.iterrows():
    line = row['url']
    match = re.search(r'/title/(tt\d+)/usercomments', line)
    if match:
        urltrain_unsup.append(match.group(1))

print(len(urltrain_unsup))

directory_path = 'aclImdb/train/unsup'
revtrain_unsup = create_dataframe_from_txt_files(directory_path, urltrain_unsup)
print(revtrain_unsup.head())

none_count = revtrain_unsup[revtrain_unsup['iid'].isnull()].shape[0]
print(f"Number of 'None' values in 'iid' column: {none_count}")

50000
                                              review        iid
0  Wow, I just rented this at Starworld video and...  tt0018515
1  I was pretty sure, when I saw that Liam, Julia...  tt0018515
2  Bambi II is one of those rare gems of a movie ...  tt0018515
3  This is truly one of the worst films I've seen...  tt0018515
4  Through rave reviews and an outstanding 7+ rat...  tt0018515
Number of 'None' values in 'iid' column: 0


In [ ]:

combined_reviews = pd.concat([revtest_neg, revtest_pos, revtrain_pos, revtrain_neg, revtrain_unsup], ignore_index=True)
print(combined_reviews.head())
combined_reviews.shape


                                              review        iid
0  Well, I set out with a few friends to see this...  tt0406816
1  A low-rent, cheaply made police thriller that'...  tt0406816
2  The beginning of the film is promising. When J...  tt0406816
3  I saw Bogard when it was released in the 70s. ...  tt0406816
4  I have been a fan of the Carpenters for a long...  tt0406816


(100000, 2)

In [ ]:
import re

def clean_text(text):
    """Removes illegal characters from text for Excel compatibility."""
    if isinstance(text, str):

        text = re.sub(r'[\000-\010]|[\013-\014]|[\016-\037]', '', text)

    return text

combined_reviews['review'] = combined_reviews['review'].apply(clean_text)

combined_reviews.to_excel('combined_reviews.xlsx', index=False)

In [ ]:
unique_iids = combined_reviews['iid'].nunique()
print(f"Number of unique iids: {unique_iids}")


Number of unique iids: 14127


In [ ]:
tms = pd.read_excel('tm_map.xlsx')
iids = tms['imdb_id']

In [ ]:
filtered_reviews = combined_reviews[combined_reviews['iid'].isin(iids)]

In [ ]:
filtered_reviews.shape

(5028, 2)

In [ ]:
unique_iids = filtered_reviews['iid'].nunique()
print(f"Number of unique iids: {unique_iids}")



Number of unique iids: 350


In [ ]:
filtered_reviews.to_excel('filtered_reviews.xlsx')

# RT reviews

In [ ]:
rtids = ['the_cabinet_of_dr_caligari', '1052609-kid', 'battleship_potemkin', 'the_gold_rush', '1043525-lost_world', '1013775-metropolis', '1008166-general', '1010978-jazz_singer', 'sunrise', 'passion_of_joan_of_arc', 'all_quiet_on_the_western_front', 'city_lights', '1006234-dracula', '1007818-frankenstein', '1016885-public_enemy', 'doctor_x', 'freaks', 'horse_feathers', 'most_dangerous_game', '1018323-scarface', 'duck_soup', '42nd_street', '1010695-invisible_man', 'island_of_lost_souls_1933', 'kennel_murder_case', '1011615-king_kong', '1038510-black_cat', 'it_happened_one_night', 'thin_man', '1000121-39_steps', 'bride_of_frankenstein', '1003499-captain_blood', 'top_hat', 'modern_times', 'partie_de_campagne', '1016756-prisoner_of_zenda', '1000355-adventures_of_robin_hood', 'bringing_up_baby', 'port_of_shadows', 'gone_with_the_wind', 'mr_smith_goes_to_washington', 'ninotchka', 'only_angels_have_wings', 'the_rules_of_the_game', '1019774-stagecoach', 'the_wizard_of_oz_1939', 'fantasia', 'grapes_of_wrath', 'great_dictator', 'his_girl_friday', '1083004-mark_of_zorro', 'philadelphia_story', 'pinocchio_1940', '1017293-rebecca', 'the_thief_of_bagdad_1940', 'citizen_kane', 'hellzapoppin', 'lady_eve', '1013139-maltese_falcon', 'sullivans_travels', 'wolf_man', '1003707-casablanca', '1003757-cat_people', '1013071-magnificent_ambersons', 'palm_beach_story', 'pride_of_the_yankees', 'i_walked_with_a_zombie', 'life_and_death_of_colonel_blimp', 'day_of_wrath_1948', 'arsenic_and_old_lace', 'double_indemnity', 'best_years_of_our_lives', '1012007-laura', 'ministry_of_fear', '1003094-brief_encounter', 'children_of_paradise', '1001902-beauty_and_the_beast', '1002352-big_sleep', 'gilda', 'its_a_wonderful_life', '1015287-notorious', 'black_narcissus', 'body_and_soul_1947', '1005412-dead_reckoning', 'out_of_the_past', 'bicycle_thieves', 'letter_from_an_unknown_woman', 'red_river', '1017346-red_shoes', 'treasure_of_the_sierra_madre', '1000253-adams_rib', 'kind_hearts_and_coronets', 'reckless_moment', 'stromboli', 'the_third_man', '1000626-all_about_eve', 'asphalt_jungle', '1002930-born_yesterday', 'cinderella', 'destination_moon', 'harvey_1950', 'in_a_lonely_place', 'los_olvidados', 'rashomon', 'sunset_boulevard', 'the_african_queen_1951', '1029112-alice_in_wonderland', 'american_in_paris', '1107426-ace_in_the_hole', '1005371-day_the_earth_stood_still', 'place_in_the_sun', 'red_badge_of_courage', 'strangers_on_a_train', '1020333-streetcar_named_desire', 'big_sky', '1046060-high_noon', 'naked_spur', 'quiet_man', 'rancho_notorious', 'singin_in_the_rain', 'band_wagon', 'beast_from_20000_fathoms', '1002332-big_heat', '1007931-from_here_to_eternity', 'sawdust_and_tinsel', 'it-came-from-outer-space', 'pickup_on_south_street', 'roman_holiday', 'wages_of_fear', 'shane', 'stalag_17', 'tokyo_story', 'i_vitelloni', 'the_war_of_the_worlds', 'barefoot_contessa', '1004906-creature_from_the_black_lagoon', '1032980-diabolique', 'dial_m_for_murder', 'godzilla_1956', 'on_the_waterfront', '1017289-rear_window', '1018047-sabrina', 'sansho_the_bailiff', '1021186-them', 'this_island_earth', 'wild_one', 'rififi', '1006416-east_of_eden', 'kiss_me_deadly', '1011818-ladykillers', 'night_of_the_hunter', 'ordet', 'picnic', 'rebel_without_a_cause', 'seven_year_itch', 'to_catch_a_thief', '1001572-bad_seed', 'bigger_than_life', 'forbidden_planet', 'giant', '1010678-invasion_of_the_body_snatchers', 'killing', '1011605-king_and_i', 'the_man_who_knew_too_much_1956', 'searchers', 'somebody_up_there_likes_me', 'a_man_escaped', 'written_on_the_wind', '1000013_12_angry_men', 'bridge_on_the_river_kwai', '1143135-forty_guns', 'funny_face', 'i_was_a_teenage_werewolf', 'incredible_shrinking_man', 'throne_of_blood', 'mon_oncle', 'curse_of_the_demon', 'paths_of_glory', 'seventh_seal', 'sweet_smell_of_success', 'witness_for_the_prosecution', 'seventh_voyage_of_sinbad', 'elevator_to_the_gallows', 'big_country', 'the_blob', 'cat_on_a_hot_tin_roof_1958', 'horror_of_dracula', '1007600-fly', 'houseboat', 'some_came_running', 'vertigo', 'anatomy-of-a-murder', 'benhur', 'brain-that-wouldnt-die', '1010417-imitation_of_life', 'north-by-northwest', '400_blows', 'rio_bravo', '1019187-sleeping_beauty', 'some_like_it_hot', 'breathless', '1001115-apartment', 'lavventura', 'dolce_vita', '1013077-magnificent_seven', 'psycho', 'rocco_and_his_brothers', '1019544-spartacus', '1022823-village_of_the_damned', 'last_year_at_marienbad', 'guns_of_navarone', 'hustler', '1099622-innocents', 'judgment_at_nuremburg', 'jules_and_jim', 'misfits', 'viridiana', 'west_side_story', 'yojimbo', 'carnival_of_souls', 'dr_no', 'lawrence_of_arabia', 'man_who_shot_liberty_valance', '1013227-manchurian_candidate', '1080265-requiem_for_a_heavyweight', 'ride_the_high_country', 'to_kill_a_mockingbird', 'what_ever_happened_to_baby_jane', '1002448-birds', '1002611-blood_feast', 'dr_strangelove', 'from_russia_with_love', 'the_leopard_1963', 'the_great_escape', '1010939-jason_and_the_argonauts', 'contempt', 'high_and_low', 'black_sabbath', 'band_of_outsiders', 'goldfinger', 'kwaidan', 'mary_poppins', 'my_fair_lady', 'the_umbrellas_of_cherbourg_1964', 'robinson_crusoe_on_mars', '1018909-shot_in_the_dark', '1050388-last_man_on_earth', 'alphaville', 'the_battle_of_algiers', 'cat_ballou', '1005279-darling', 'sound_of_music', 'blow_up_1966', 'the_good_the_bad_and_the_ugly', '1007003-fahrenheit_451', '1013162-man_for_all_seasons', 'plague_of_the_zombies', 'seconds', 'face_of_another', 'whos_afraid_of_virginia_woolf', 'belle_de_jour', 'bonnie_and_clyde', 'cool_hand_luke', 'dirty_dozen', 'graduate', '1010448-in_cold_blood', '1031385-jungle_book', '1016479-point_blank', '2001_a_space_odyssey', 'barbarella', 'bullitt', 'faces', 'funny_girl', 'je_taime_je_taime', 'night_of_the_living_dead', '1015380-odd_couple', '1016397-planet_of_the_apes', '1016819-producers', 'rosemarys_baby', 'beatles_the_yellow_submarine', '1003318-butch_cassidy_and_the_sundance_kid', 'once_upon_a_time_in_the_west', 'easy_rider', 'midnight_cowboy', 'true_grit', 'bird_with_the_crystal_plumage', '1059489-wild_bunch', 'z', 'airport', 'beyond_the_valley_of_the_dolls', 'the_conformist', 'little_big_man', 'mash', 'patton', 'performance', 'vampyros_lesbos', 'blood_on_satans_claw', 'clockwork_orange', 'dirty_harry', '1076721-get_carter', 'harold_and_maude', 'last_picture_show', 'mccabe_and_mrs_miller', 'omega_man', 'a_bay_of_blood', 'silent_running', 'straw_dogs', 'el_topo', 'twolane-blacktop', 'cabaret', 'the_discreet_charm_of_the_bourgeoisie', 'deliverance', 'the_godfather', 'high_plains_drifter', '1016584-poseidon_adventure', 'solaris_1976', 'tales_from_the_crypt_1972', 'ulzanas_raid', 'whats_up_doc', 'american_graffiti', 'asphyx', 'badlands', 'dont_look_now', 'enter_the_dragon', 'spirit_of_the_beehive', 'exorcist', '1031440-long_goodbye', 'the_mother_and_the_whore', 'mean_streets', 'paper_moon', 'pat_garrett_and_billy_the_kid', 'phase_iv_1974', 'serpico', 'sisters_1973', 'soylent_green', '1020130-sting', 'last_tango_in_paris', 'way_we_were', 'westworld', 'black_christmas_1974', 'blazing_saddles', 'bring_me_the_head_of_alfredo_garcia', 'chinatown', 'the_conversation', 'holy_mountain', 'monty_python_and_the_holy_grail', 'parallax_view', '1021112-texas_chainsaw_massacre', 'towering_inferno', 'woman_under_the_influence', 'young_frankenstein', 'barry_lyndon', 'boy_and_his_dog', 'death_race_2000', 'dog_day_afternoon', 'jaws', 'love_and_death', 'man_who_would_be_king', 'nashville', 'one_flew_over_the_cuckoos_nest', 'the-passenger-professione-reporter', 'deep_red_1975', 'rocky_horror_picture_show', 'shampoo', 'three_days_of_the_condor', '1021648-tommy', 'all_the_presidents_men', '1001280-assault_on_precinct_13', '1001567-bad_news_bears', 'bugsy_malone', '1003625-carrie', 'eraserhead', '1008968-grizzly', 'cross_of_iron', 'logans_run', 'man_who_fell_to_earth', 'network', '1015517-omen', 'outlaw_josey_wales', '1017776-rocky', 'shootist', 'taxi_driver', 'annie_hall', 'close_encounters_of_the_third_kind', '1007839-freaky_friday', 'the_sentinel_1977', 'slap_shot', 'smokey_and_the_bandit', '1020662-suspiria', 'wizards', 'days_of_heaven', 'the_deer_hunter', 'five_deadly_venoms', '1009113-halloween', '1009427-heaven_can_wait', '1009645-hills_have_eyes', '1010679-invasion_of_the_body_snatchers', 'midnight_express', 'national_lampoons_animal_house', '1016359-piranha', 'up_in_smoke_1978', 'alien', 'all_that_jazz', 'the_amityville_horror', 'apocalypse_now', 'being_there', 'breaking_away', 'brood', 'cannibal_holocaust', 'jerk', 'monty_pythons_life_of_brian', 'mad_max', 'manhattan', 'phantasm', '1023205-warriors', '1023461-when_a_stranger_calls', 'airplane', 'alligator', 'altered_states', 'american_gigolo', 'blues_brothers', 'caddyshack', 'changeling', '1006527-elephant_man', '1007517-flash_gordon', 'the_fog_1980', 'friday_the_13th_part_1', 'heavens_gate', 'ordinary_people', '1016834-prom_night', 'raging_bull', '1018315-scanners', 'shining', 'stir_crazy', 'time_bandits', '1002830-body_heat', 'das_boot', 'chariots_of_fire', 'clash_of_the_titans', 'conan_the_barbarian', '1006717-escape_from_new_york', 'excalibur', 'the_funhouse_1981', 'pieces', 'my_bloody_valentine', 'outland', 'raiders_of_the_lost_ark', 'reds', 'wolfen', 'blade_runner', 'creepshow', 'dark_crystal', 'et_the_extraterrestrial', 'the-evil-dead', 'fast_times_at_ridgemont_high', 'fitzcarraldo', 'gandhi', 'last_unicorn', '1016513-poltergeist', 'sophies_choice', 'star_trek_ii_the_wrath_of_khan', '1021244-thing', 'tootsie', 'tron', 'verdict', '1004151-christmas_story', 'cujo', 'dead_zone', '1011623-king_of_comedy', 'krull', 'local_hero', 'lone_wolf_mcquade', 'national_lampoons_vacation', '1017641-risky_business', 'scarface', 'testament', 'videodrome', 'year_of_living_dangerously', 'zelig', '2010_the_year_we_make_contact', 'amadeus', 'beverly_hills_cop', 'blood_simple', 'the_brother_from_another_planet', 'chud', '1004047-children_of_the_corn', '1006364-dune', 'friday-the-13th-the-final-chapter', 'ghostbusters', 'gremlins', 'karate_kid', 'killing_fields', 'natural', 'night_of_the_comet', 'nightmare_on_elm_street', 'paris_texas', 'purple_rain', 'repo_man', 'revenge_of_the_nerds', '1018989-silent_night_deadly_night', 'sixteen_candles', 'splash', 'terminator', 'this_is_spinal_tap', 'the_neverending_story_1984', 'back_to_the_future', '1003033-brazil', 'breakfast_club', 'cocoon', '1004567-commando', '1005360-day_of_the_dead', '1007910-fright_night', 'ghoulies', 'goonies', 'ladyhawke', '1012164-legend', 'lost_in_america', 'ran', 'reanimator', '1017352-red_sonja', 'return_of_the_living_dead', 'stephen_kings_silver_bullet', 'teen_wolf', 'toxic_avenger', 'crocodile_dundee', '1000617-aliens', 'big_trouble_in_little_china', 'blue_velvet', 'color_of_money', 'down_by_law', 'ferris_buellers_day_off', 'flight_of_the_navigator', '1007602-fly', 'highlander', 'hoosiers', '1036052-come_and_see', 'labyrinth', 'maximum_overdrive', 'night_of_the_creeps', '9_12_weeks', 'peggy_sue_got_married', 'platoon', 'pretty_in_pink', 'stand_by_me_1986', 'top_gun', 'bad_taste', 'broadcast_news', 'the_dead', '1007141-fatal_attraction', 'full_metal_jacket', 'gate', 'hellraiser', 'hidden', 'lethal_weapon', 'monster_squad', 'moonstruck', '1014793-near_dark', 'predator', 'princess_bride', 'raising_arizona', '1017712-robocop', 'running_man', 'spaceballs', 'stand_and_deliver', 'untouchables', 'akira', 'alien_nation', 'beetlejuice', 'big', 'bull_durham', 'childs_play', 'coming_to_america', 'cry_in_the_dark', '1005399-dead_heat', 'die_hard', 'distant_voices_still_lives', 'eight_men_out', 'a_fish_called_wanda', '1009096-hairspray', 'midnight_run', 'mississippi_burning', 'cinema_paradiso', 'pumpkinhead', 'rain_man', 'they_live', 'my_neighbor_totoro_1988_2', 'who_framed_roger_rabbit', 'willow', 'adventures_of_baron_munchausen', '1001781-batman', 'born_on_the_fourth_of_july', 'crimes_and_misdemeanors', 'dead_poets_society', '1032434-killer', 'do_the_right_thing', 'dream_a_little_dream', 'field_of_dreams', '1008415-glory', 'heathers', 'indiana_jones_and_the_last_crusade', '1011487-kickboxer', 'the_little_mermaid_1989', 'major_league', 'my_left_foot', 'parenthood', 'pet_sematary', '1017666-road_house', 'say_anything', 'when_harry_met_sally', 'whos_harry_crumb', 'sex_lies_and_videotape', '1032970-awakenings', '1034188-cyrano_de_bergerac', 'dances_with_wolves', 'darkman', 'edward_scissorhands', '1032176-goodfellas', 'hardware', 'home_alone', 'metropolitan', 'millers_crossing', 'misery', 'la_femme_nikita', 'pretty_woman', 'quick_change', 'total_recall', 'tremors', 'the_witches', 'barton_fink', 'beauty_and_the_beast_1991', 'boyz_n_the_hood', 'delicatessen', '1037864-father_of_the_bride', 'hook', 'la_story', 'one_false_move', 'point_break', 'silence_of_the_lambs', 'terminator_2_judgment_day', 'thelma_and_louise', 'once_upon_a_time_in_china', 'disneys_aladdin', 'basic_instinct', 'dead_alive', 'buffy_the_vampire_slayer', 'candyman', 'crying_game', 'few_good_men', 'glengarry_glen_ross', 'supercop', '1040678-last_of_the_mohicans', 'a_league_of_their_own', 'lorenzos_oil', '1042135-malcolm_x', 'mighty_ducks', 'player', 'reservoir_dogs', 'under_siege', '1041911-unforgiven', 'army_of_darkness', 'carlitos_way', '1046227-cool_runnings', 'dave', 'dazed_and_confused', '1046129-fugitive', 'groundhog_day', 'in_the_line_of_fire', 'in_the_name_of_the_father', 'jurassic_park', 'mrs_doubtfire', 'nightmare_before_christmas', 'philadelphia', 'remains_of_the_day', 'rudy', 'sandlot', 'schindlers_list', 'short_cuts', 'sleepless_in_seattle', 'tombstone', 'true_romance', 'whats_love_got_to_do_with_it', 'ace_ventura_pet_detective', 'bullets_over_broadway', 'chungking_express', 'clerks', 'crow', 'ed_wood', 'exotica', 'forrest_gump', 'four_weddings_and_a_funeral', 'heavenly_creatures', 'the_last_seduction', 'the_lion_king', 'pulp_fiction', 'quiz_show', 'the-secret-of-roan-inish', 'shawshank_redemption', 'speed_1994', 'stargate', 'true_lies', 'burnt_by_the_sun', 'wes_cravens_new_nightmare', '1053942-wolf', '1068307-addiction', 'apollo_13', '1065598-babe', '1062483-bad_boys', 'before_sunrise', 'billy_madison', '1065684-braveheart', '1067987-casino', 'city_of_lost_children', 'clueless', 'dead_man', '1068779-dead_man_walking', 'die_hard_with_a_vengeance', 'friday', 'heat_1995', 'kids', 'leaving_las_vegas', '1069339-restoration', 'safe', 'strange_days', 'tommy_boy', 'toy_story', '12_monkeys', 'usual_suspects', 'big_night', 'birdcage', 'bottle_rocket', 'bound', 'dont_be_a_menace', 'english_patient', 'fargo', 'from_dusk_till_dawn', 'happy_gilmore', 'i_shot_andy_warhol', '1071806-independence_day', 'james_and_the_giant_peach', 'jerry_maguire', '1072385-kingpin', '1074022-lone_star', 'lost_highway', '1072107-matilda', 'mission_impossible', 'people_vs_larry_flynt', '1074298-ransom', '1072011-rock', '1074316-scream', 'secrets_and_lies', 'shine', 'sling_blade', 'trainspotting', '1071167-twister', 'waiting_for_guffman', 'apostle', 'austin_powers_international_man_of_mystery', 'the_big_lebowski', 'boogie_nights', '1084398-life_is_beautiful', '1078021-contact', 'dark_city', 'devils_advocate', 'fifth_element', 'gattaca', 'good_will_hunting', 'grosse_pointe_blank', 'hard_eight', 'i_know_what_you_did_last_summer', 'ice_storm', 'jackie_brown', 'la_confidential', 'men_in_black', 'mimic', 'princess_mononoke_1999', 'as_good_as_it_gets', 'three_kings', 'sweet_hereafter', 'taste_of_cherry', 'simple_plan', 'titanic', 'toy_story_2', 'truman_show', 'american_history_x', 'being_john_malkovich', '1083484-blade', 'a_bugs_life', 'chicken_run', 'enemy_of_the_state', 'eyes_wide_shut', 'fear_and_loathing_in_las_vegas', 'frida', 'gods_and_monsters', 'green_mile', 'lock_stock_and_two_smoking_barrels', 'the_lord_of_the_rings_the_fellowship_of_the_ring', 'mulan', '1083436-1083436-out_of_sight', 'rush_hour', 'saving_private_ryan', '1084146-thin_red_line', 'wag_the_dog', 'xmen', 'existenz', 'ronin', '1093579-man_on_the_moon', 'shrek', 'election', '1084153-elizabeth', '1083659-rounders', 'rushmore', 'iron_giant', 'theres_something_about_mary', 'matrix', 'talented_mr_ripley', 'fight_club', 'shakespeare_in_love', 'pi', 'training_day', 'insider', 'american_psycho', 'spiderman', 'any_given_sunday', '1095420-high_fidelity', '1084175-happiness', '1090759-deep_blue_sea', 'topsyturvy', 'office_space', '1094723-celebration', 'sweet_and_lowdown', 'virgin_suicides', 'story_of_us', 'cast_away', 'ghost_world', 'bringing_out_the_dead', 'after_life', 'limey', 'waking_ned_devine', 'mulholland_dr', 'hellboy', 'the_lord_of_the_rings_the_return_of_the_king', 'the_lord_of_the_rings_the_two_towers', 'sixth_sense', 'american_beauty', 'boys_dont_cry', 'girl_interrupted', 'gladiator', 'magnolia', 'galaxy_quest', 'requiem_for_a_dream', 'minority_report', '1103281-traffic', 'pollock', 'wonder_boys', 'all_about_my_mother', 'blair_witch_project', 'shadow_of_the_vampire', 'crouching_tiger_hidden_dragon', 'erin_brockovich', 'final_destination', 'zoolander', 'monsters_inc', 'sexy_beast', 'together_2001', 'you_can_count_on_me', 'children_of_men', 'snatch', 'memento', 'ginger_snaps', 'remember_the_titans', 'amelie', '1097259-road_trip', 'sixth_day', 'unbreakable', 'best_in_show', '1109257-others', 'equilibrium', 'oceans_eleven', 'chocolat', 'harry_potter_and_the_sorcerers_stone', 'waking_life', 'wet_hot_american_summer', 'spirited_away', 'y_tu_mama_tambien', 'donnie_darko', 'before_night_falls', 'in_the_bedroom', 'ali', 'hedwig_and_the_angry_inch', 'billy_elliot', 'pianist', 'devils_backbone', 'about_schmidt', 'panic_room', 'bourne_identity', 'vanilla_sky', 'catch_me_if_you_can', 'frailty', 'monsoon_wedding', 'the_royal_tenenbaums', 'battle_royale', 'kill_bill_vol_1', 'hours', 'secretary', 'about_a_boy', 'we_were_soldiers', 'love_liza', 'signs', 'talk_to_her', '28_days_later', 'chronicles_of_riddick', 'far_from_heaven', 'ring', 'chicago', 'morvern_callar', 'old_school', 'harry_potter_and_the_prisoner_of_azkaban', 'american_splendor', 'master_and_commander_the_far_side_of_the_world', 'wallace_and_gromit_the_curse_of_the_were_rabbit', 'raising_victor_vargas', 'city_of_god', 'incredibles', '1127787-big_fish', 'elf', 'pirates_of_the_caribbean_the_curse_of_the_black_pearl', 'mystic_river', 'coraline', 'seabiscuit', 'garden_state', 'lost_in_translation', '13_going_on_30', 'eternal_sunshine_of_the_spotless_mind', 'aviator', 'station_agent', 'monster', 'i_robot', 'howls_moving_castle', 'miracle', 'ray', 'i_heart_huckabees', 'anchorman', 'walk_the_line', 'downfall', 'dawn_of_the_dead', 'elephant', 'chronicles_of_narnia_lion_witch_wardrobe', 'shaun_of_the_dead', 'happy_feet', 'squid_and_the_whale', 'im_not_there_suppositions_on_a_film_concerning_dylan', 'osama', 'collateral', 'sea_inside', 'iron_man', 'kung_fu_hustle', 'napoleon_dynamite', 'sideways', 'layer_cake', 'closer', 'the_return', 'mean_girls', '1151898-capote', 'serenity', 'casino_royale', 'tropical_malady', 'before_sunset', 'ratatouille', 'house_of_flying_daggers', 'where_the_wild_things_are', 'saw', 'idiocracy', 'brokeback_mountain', 'friday_night_lights', 'maria_full_of_grace', 'primer', 'hotel_rwanda', 'history_of_violence', 'the_island_2005', 'the_diving_bell_and_the_butterfly_2007', 'sin_city', '1152954-new_world', 'little_children', 'the_lives_of_others', 'million_dollar_baby', 'scanner_darkly', '40_year_old_virgin', 'departed', 'flightplan', 'hustle_and_flow', 'broken_flowers', 'talladega_nights_the_ballad_of_ricky_bobby', 'match_point', '300', 'prairie_home_companion', 'stranger_than_fiction', 'hot_fuzz', 'thank_you_for_smoking', '1197696-fantastic_mr_fox', 'v_for_vendetta', 'descent', 'toy_story_3', 'the_queen', 'behind_the_mask_the_rise_of_leslie_vernon', 'slither', 'lincoln_2011', 'borat', 'assassination_of_jesse_james_by_the_coward_robert_ford', 'zodiac', 'shazam', 'sunshine', 'little_miss_sunshine', 'hostel', 'wonder_woman_2017', 'break_up', '42_2013', 'life-of-pi', 'the_last_king_of_scotland', 'dead_silence', 'the_equalizer_2013', 'the_death_of_mr_lazarescu', 'pans_labyrinth', 'nacho_libre', 'inland_empire', 'enchanted', 'john_rambo', '28_weeks_later', 'the_orphanage', 'half_nelson', 'the_host_2007', 'the_dark_knight', 'there_will_be_blood', 'ex_machina', 'no_country_for_old_men', 'the_tree_of_life_2011', 'knocked_up', 'crank', 'i_am_legend', 'prestige', 'the_strangers', 'stardust', '1174279-wanted', 'vicky_cristina_barcelona', 'avatar', 'into_the_wild', 'american_gangster', 'eastern_promises', 'drive_2011', '1196003-princess_and_the_frog', 'in_bruges', 'star_trek_11', 'forgetting_sarah_marshall', 'persepolis', 'interstellar_2014', 'world-war-z', 'superbad', '1193743-step_brothers', 'walk_hard', 'paprika', 'elite_squad', '1212694-blind_side', 'the_hurt_locker', 'how_to_train_your_dragon', 'the_road_2009', 'once', 'harry_potter_and_the_deathly_hallows_part_1', 'taken', 'revolutionary_road', 'the_fighter_2011', 'straight_story', 'machete', 'the_wolf_of_wall_street_2013', 'slumdog_millionaire', '1208173-splice', 'a_serious_man', '500_days_of_summer', 'argo_2012', 'martyrs', 'the_power_of_the_dog', '4_months_3_weeks_and_2_days', 'the_bands_visit', 'suspiria_2018', 'infinity_pool_2023', 'silver_linings_playbook', 'up', 'ma_raineys_black_bottom', 'the_invisible_man_2020', 'cloverfield', 'boyhood', 'the_class', 'nope', 'foxcatcher', 'my_heart_cant_beat_unless_you_tell_it_to', 'paddington_2014', '10010667-hangover', 'the_wrestler', 'licorice_pizza', 'drag_me_to_hell', 'district_9', 'the_killer_2022', 'let_the_right_one_in', 'no_sudden_move', 'zombieland', 'the_house_of_the_devil_2009', 'an_education', '10_cloverfield_lane', 'the_banshees_of_inisherin', 'moon', 'prey_2022', 'agora', '1216813-triangle', 'first_man', 'ip_man', 'pontypool', 'in_the_loop', '21_jump_street_2011', '1217700-kick_ass', 'the_cabin_in_the_woods', 'x_men_first_class', 'looper', 'dead_snow', 'easy-a', 'plan_b_2021', 'the_expendables', 'in_the_heights_2021', 'past_lives', 'brian_and_charles', 'never_let_me_go_2010', 'tinker_tailor_soldier_spy', 'four_lions', 'dredd', 'inception', 'the_hunger_games', 'mad_max_fury_road', 'prisoners_2013', 'it_2017', '10012136-winters_bone', 'asteroid_city', 'deadpool', '13_assassins_2011', 'martha_marcy_may_marlene', 'under_the_skin_2013', 'tar_2022', 'gravity_2013', 'the_conjuring', 'buried', 'monsters-2010', 'bridesmaids_2011', 'attack_the_block', 'oblivion_2013', 'booksmart', 'the_lego_movie', 'hellbender', 'the_kings_speech', 'mad_god', 'barbie', 'a_star_is_born_2018', 'oppenheimer_2023', '127_hours', 'another_earth', 'rye_lane', 'insidious', 'the_innkeepers', 'fast_five', 'midnight_in_paris', 'absentia_2012', 'rubber', 'bleed_for_this', 'live_die_repeat_edge_of_tomorrow', 'the_perks_of_being_a_wallflower', 'the_revenant_2015', 'pacific_rim_2013', 'the_intouchables', 'the_awakening_2011', 'snowpiercer', 'wreck_it_ralph', 'her', 'black_panther_2018', 'django_unchained_2012', 'blade_runner_2049', 'the_edge_of_seventeen', 'the_raid_redemption', 'ford_v_ferrari', 'aftersun', 'resolution_2013', 'rush_2013', 'toy_story_4', 'guardians_of_the_galaxy', 'all_is_lost_2013', '12_years_a_slave', 'europa_report', 'upstream_color', 'inside_out_2015', 'vhs', 'the_hunt_2013', 'hacksaw_ridge', 'before_midnight_2013', 'big_hero_6', 'spider_man_homecoming', 'gone_girl', 'frozen_2013', 'we_are_what_we_are_2013', 'my_life_as_a_zucchini', 'the_babadook', 'a_girl_walks_home_alone_at_night', 'mississippi_grind', 'blue_ruin', 'coco_2017', 'oculus', 'predestination', 'the_invitation', 'atomic_blonde_2017', 'creep_2014', 'bone_tomahawk', 'wyrmwood_road_of_the_dead_2015', 'arrival_2016', 'crimson_peak', 'birdman_2014', 'the_tale_of_the_princess_kaguya', 'hell_or_high_water', 'whiplash_2014', 'at_the_devils_door', 'sea_fever_2020', 'wolfcop', 'petes_dragon_2016', 'starry_eyes', 'annihilation', 'kingsman_the_secret_service', 'when_animals_dream', 'furious_7', 'the_old_man_and_the_gun', 'coherence_2013', 'john_wick', 'ad_astra', 'soul', 'the_jungle_book_2016', 'goodnight_mommy', 'crazy_rich_asians', 'it_follows', 'sully', 'the_autopsy_of_jane_doe', 'logan_2017', 'annabelle', 'the_taking_of_deborah_logan_2014', 'spring_2015', 'sicario_2015', 'a_monster_calls', 'what_we_do_in_the_shadows', 'the_lobster', 'captain_america_civil_war', 'thor_ragnarok_2017', 'we_are_still_here', 'the_disaster_artist', 'moana_2016', 'girls_trip', 'west_side_story_2021', 'incredibles_2', 'in_a_valley_of_violence', 'the_martian', 'the_bfg_2016', 'experimenter', 'rogue_one_a_star_wars_story', 'la_la_land', 'the_spine_of_night', 'baby_driver', 'popstar_never_stop_never_stopping', 'okja', 'the_monster_2016', 'the_endless', 'manchester_by_the_sea', 'the_shallows', 'green_room_2016', 'frankenstein_2015', 'they_look_like_people', 'the_lego_batman_movie', 'captain_marvel', 'avengers_infinity_war', 'avengers_endgame', 'dont_breathe_2016', 'the_birth_of_a_nation_2016', 'widows_2018', 'the_witch_2016', 'the_night_eats_the_world', 'kubo_and_the_two_strings_2016', 'paddington_2', 'overlord_2018', 'the_girl_with_all_the_gifts', 'christopher_robin', 'can_you_ever_forgive_me', 'spider_man_into_the_spider_verse', 'loving_2016', 'colossal', 'the_death_of_stalin', 'it_comes_at_night', 'high_life_2019', 'hidden_figures', 'southbound', 'raw_2017', 'moonlight_2016', 'marjorie_prime', 'dunkirk_2017', 'hush_2016', 'three_billboards_outside_ebbing_missouri', 'get_out', 'color_out_of_space', 'the_favourite_2018', 'isle_of_dogs_2018', 'mother_2017', 'the_wailing', 'happy_death_day', 'colette_2018', 'logan_lucky', 'life_2017', 'the_big_sick', 'ghost_stories_2018', 'a_futile_and_stupid_gesture', 'i_tonya', 'the_shape_of_water_2017', 'the_beguiled', 'brawl_in_cell_block_99', 'sorry_to_bother_you_2018', 'train_to_busan', 'the_killing_of_a_sacred_deer', 'call_me_by_your_name', 'uncut_gems', 'you_were_never_really_here', 'phantom_thread', 'ralph_breaks_the_internet', 'the_empty_man', 'columbus_2017', 'before_we_vanish', 'roma_2018', 'apostle_2018', 'the_rider', 'a_ghost_story', 'i_am_mother', 'dragged_across_concrete', 'upgrade_2018', 'a_quiet_place_2018', 'everything_everywhere_all_at_once', 'the_vast_of_night', 'us_2019', 'mirai', 'mandy_2018', 'eighth_grade', 'destroyer', 'blackkklansman', 'aniara', 'marriage_story_2019', 'hereditary', 'prospect_2018', 'his_house', '1917_2019', 'midsommar', 'dual_2022', 'synchronic', 'relic', 'emma_2020', 'first_cow', 'the_green_knight', 'spider_man_across_the_spider_verse', 'palm_springs_2020', 'the_night_house', 'the_world_to_come', 'nomadland', 'judas_and_the_black_messiah']

In [ ]:
len(rtids)

1366

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "rotten_tomatoes_movie_reviews.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

<ipython-input-4-9ec5795cd986>:10: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  df = kagglehub.load_dataset(


100%|██████████| 145M/145M [00:04<00:00, 35.0MB/s]

Extracting zip of rotten_tomatoes_movie_reviews.csv...


First 5 records:                                   id  reviewId creationDate       criticName  \
0                            beavers   1145982   2003-05-23  Ivan M. Lincoln   
1                         blood_mask   1636744   2007-06-02    The Foywonder   
2  city_hunter_shinjuku_private_eyes   2590987   2019-05-28     Reuben Baron   
3  city_hunter_shinjuku_private_eyes   2558908   2019-02-14      Matt Schley   
4                 dangerous_men_2015   2504681   2018-08-29        Pat Padua   

   isTopCritic originalScore reviewState                 publicatioName  \
0        False         3.5/4       fresh  Deseret News (Salt Lake City)   
1        False           1/5      rotten                  Dread Central   
2        False           NaN       fresh                            CBR   
3        False         2.5/5      rotten                    Japan Times   
4        False           NaN       fresh                          DCist   

                                          reviewTex

In [ ]:
rtr = df.copy()

In [ ]:
rtrf = rtr[rtr['id'].isin(rtids)]

In [ ]:
unique_id_count = rtrf['id'].nunique()
print(f"The number of unique 'id' values in rtrf is: {unique_id_count}")


The number of unique 'id' values in rtrf is: 1349


In [ ]:
rtrf.to_excel('rt_reviews_filtered.xlsx', index=False)

In [ ]:
rtrf.shape

(159224, 11)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter


# IMDB 7M reviews

In [ ]:

file_path = "part-06.json"

# Load the latest version
j6 = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ebiswas/imdb-review-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", j6.head())

<ipython-input-311-c6d7617e25fc>:4: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  j6 = kagglehub.load_dataset(


100%|██████████| 221M/221M [00:06<00:00, 34.0MB/s]

Extracting zip of part-06.json...


First 5 records:    review_id         reviewer               movie  rating  \
0  rw0099142          ed.wenn  Le Samouraï (1967)     9.0   
1  rw0099143  the red duchess  Le Samouraï (1967)    10.0   
2  rw0099144      esteban1747  Le Samouraï (1967)     7.0   
3  rw0099146    youroldpaljim  Le Samouraï (1967)     NaN   
4  rw0099147           tasgal  Le Samouraï (1967)     5.0   

                                      review_summary        review_date  \
0                                   Cooler Than Cool   26 November 2000   
1  Along with 'The Wizard of Oz', the supreme fil...    7 December 2000   
2           Jeff Costello, a nearly perfect gangster  20 September 2001   
3                                        Paris envy?      24 April 2002   
4         Stylish and cluelessly silly. Inoffensive.    11 January 2003   

   spoiler_tag                                      review_detail     helpful  
0            0  Surely one of the suavest movies ever made. Cr...     [8, 21]  
1    

In [ ]:
import pandas as pd
names = pd.read_excel('names.xlsx')

In [ ]:
names.head()

,a
0,The Cabinet of Dr. Caligari (1920)
1,The Kid (1921)
2,Nosferatu: A Symphony of Horror (1922)
3,Safety Last! (1923)
4,Sherlock Jr. (1924)


In [ ]:
j1 = df

In [ ]:
j4.shape

(1019000, 9)

In [ ]:
jf6 = j6[j6['movie'].isin(names['a'])]

In [ ]:
jf6.shape

(103511, 9)

In [ ]:
jf6 = jf6[['movie', 'review_summary', 'review_detail', 'helpful']]

In [ ]:
jf6.shape

(103511, 4)

In [ ]:
import re
def clean_illegal_characters(df, columns):
    """
    Removes specific illegal characters from specified columns in a DataFrame.

    Args:
        df: The input DataFrame.
        columns: A list of column names to clean.

    Returns:
        The cleaned DataFrame.
    """
    # Define a regex pattern to match illegal characters (control characters and non-printable characters)
    # Note that this is just a starting point and may need to be customized further
    illegal_characters_re = re.compile(r'[\000-\037]')  # Match control characters and other illegal char

    for column in columns:
        # Apply the cleaning function to each cell in the specified columns
        df[column] = df[column].astype(str).apply(lambda x: illegal_characters_re.sub('', x))

    return df

# Clean the 'review_summary' and 'review_detail' columns
jf4 = clean_illegal_characters(jf4, ['review_summary', 'review_detail'])

jf4.to_excel('jf4.xlsx', index=False)

Exception ignored in: <function ZipFile.__del__ at 0x7e0f2375a0c0>
Traceback (most recent call last):
  File "/usr/lib/python3.11/zipfile.py", line 1895, in __del__
    self.close()
  File "/usr/lib/python3.11/zipfile.py", line 1912, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


In [ ]:
jf6.to_excel('jf6.xlsx', index=False)

KeyboardInterrupt: 

In [ ]:
import pandas as pd

In [ ]:
jf1 = pd.read_excel('jf1.xlsx')
jf2 = pd.read_excel('jf2.xlsx')
jf3 = pd.read_excel('jf3.xlsx')
jf4 = pd.read_excel('jf4.xlsx')
jf5 = pd.read_excel('jf5.xlsx')
jf6 = pd.read_excel('jf6.xlsx')

In [ ]:
bigj = pd.concat([jf1, jf2, jf3, jf4, jf5, jf6])

In [ ]:
bigjj = bigj.copy()

In [ ]:
bigjj.head()

,movie,review_summary,review_detail,helpful
0,All About Eve (1950),Amazing,Having seen this film for the first time today...,"['0', '1']"
1,500 Days of Summer (2009),"Cutie indie, just boring enough not to rise ab...",Ah Indies done by brainerds who rever the verb...,"['0', '1']"
2,Memento (2000),"Complicated, nice story.",It's very complex and complicated. The whole f...,"['5', '7']"
3,Mean Girls (2004),Fabulous and Funny at the same time - THIS REV...,If your looking for a movie that you could pot...,"['1', '1']"
4,Apocalypse Now (1979),Boooooooring,"Can't feel my legs, wasted 3 hrs for nothing.....","['4', '8']"


In [ ]:
bigj.shape

(725934, 4)

In [ ]:
bigj['movie'].nunique()

1480

In [ ]:
ewr_raw = pd.read_excel('updated_ewg.xlsx')

In [ ]:
import pandas as pd

def convert_dataframe_to_dict(df):
    """Converts a Pandas DataFrame to a dictionary, ignoring NaN values."""
    result_dict = {}
    for col in df.columns:

        values = df[col].astype(str).tolist()
        cleaned_values = [val for val in values if val != 'nan']
        result_dict[col] = cleaned_values
    return result_dict

ewr = convert_dataframe_to_dict(ewr_raw)
ewr


{'Skepticism': ['skepticism',
  'skeptic',
  'disbelieve',
  'incredulous',
  'incredulity',
  'dubious',
  'suspicio',
  'mistrust',
  'distrust'],
 'Serenity': ['serenity',
  'serene',
  'tranquil',
  'repose',
  'equanimity',
  'placid',
  'harmony'],
 'Fear': ['fear',
  'afraid',
  'alarming',
  'terrify',
  'terrified',
  'frighten',
  'panic',
  'alarmed',
  'petrifying',
  'petrified',
  'scary'],
 'Disgust': ['disgust',
  'repulse',
  'nauseat',
  'revolt',
  'sicken',
  'appall',
  'loath',
  'repugnant',
  'detestable'],
 'Solidarity': ['solidarity',
  'unite',
  'together',
  'allied',
  'cohesive',
  'kinship'],
 'Elation': ['elation',
  'elate',
  'ecstatic',
  'jubilant',
  'overjoy',
  'rapturous'],
 'Envy': ['envy', 'jealous', 'covet', 'begrudge', 'spite', 'jealousy'],
 'Puzzlement': ['puzzlement',
  'puzzle',
  'confound',
  'baffle',
  'perplex',
  'mystify',
  'bewilder',
  'stupefy'],
 'Recklessness': ['recklessness', 'reckless', 'impulse', 'daredevil', 'brash'],
 '

In [ ]:
ititlemap = pd.read_excel('ititle_to_iid.xlsx')

In [ ]:

ititlemap = ititlemap.rename(columns={'ty': 'movie'})

In [ ]:
bigjjm = pd.merge(bigjj, ititlemap, on='movie', how='left')


In [ ]:

nan_tconst_count = bigjjm['tconst'].isna().sum()
print(f"Number of rows where 'tconst' is NaN: {nan_tconst_count}")


Number of rows where 'tconst' is NaN: 0


In [ ]:
big_r = bigjjm[['tconst', 'review_detail']]

In [ ]:
big_r = big_r.rename(columns={'review_detail': 'r'})

In [ ]:
big_r.head()

,tconst,r
0,tt0042192,Having seen this film for the first time today...
1,tt1022603,Ah Indies done by brainerds who rever the verb...
2,tt0209144,It's very complex and complicated. The whole f...
3,tt0377092,If your looking for a movie that you could pot...
4,tt0078788,"Can't feel my legs, wasted 3 hrs for nothing....."


In [ ]:
rt_r = pd.read_excel('clean_rt_reviews.xlsx')

In [ ]:
rt_r = rt_r.rename(columns={'imdb_id': 'tconst', 'reviewText': 'r'})

In [ ]:
rt_r.head()

,tconst,r
0,tt0265343,Filled with as much drama and emotion as there...
1,tt0265343,This comedy may not seem like your typical Bol...
2,tt0265343,&#91;A&#93; whirl of color and music&#44; fami...
3,tt0265343,MONSOON WEDDING is a wonderful film in which m...
4,tt0265343,A celebration of life that acknowledges how fa...


In [ ]:
fr_r = pd.read_excel('filtered_reviews350.xlsx')

In [ ]:
fr_r = fr_r.rename(columns={'iid': 'tconst', 'review': 'r'})

In [ ]:
all_r = pd.concat([big_r, rt_r, fr_r])

In [ ]:
all_r.shape

(882563, 2)

In [ ]:
all_r['tconst'].nunique()

1570

In [ ]:
all_r = all_r.drop_duplicates(subset=['tconst', 'r'])

print(all_r.shape)
print(all_r['tconst'].nunique())


(878173, 2)
1570


In [ ]:
def count_feeling_words(review, feeling_words):
    """Counts the occurrences of feeling words in a review."""
    count = 0
    review_lower = review.lower()
    for word in feeling_words:
        if review_lower.count(word) > 0:
            return 1
    return 0

In [ ]:
count = 0
feeling_counts = {}
for tconst in all_r['tconst'].unique():
    feeling_counts[tconst] = {}
    reviews_for_tconst = all_r[all_r['tconst'] == tconst]['r']
    for feeling, words in ewr.items():
        total_feeling_count = 0
        for review in reviews_for_tconst:
          if isinstance(review, str):
            total_feeling_count += count_feeling_words(review, words)
          count += 1
          print(count)
        feeling_counts[tconst][feeling] = total_feeling_count


Streaming output truncated to the last 5000 lines.
43903651
43903652
43903653
43903654
43903655
43903656
43903657
43903658
43903659
43903660
43903661
43903662
43903663
43903664
43903665
43903666
43903667
43903668
43903669
43903670
43903671
43903672
43903673
43903674
43903675
43903676
43903677
43903678
43903679
43903680
43903681
43903682
43903683
43903684
43903685
43903686
43903687
43903688
43903689
43903690
43903691
43903692
43903693
43903694
43903695
43903696
43903697
43903698
43903699
43903700
43903701
43903702
43903703
43903704
43903705
43903706
43903707
43903708
43903709
43903710
43903711
43903712
43903713
43903714
43903715
43903716
43903717
43903718
43903719
43903720
43903721
43903722
43903723
43903724
43903725
43903726
43903727
43903728
43903729
43903730
43903731
43903732
43903733
43903734
43903735
43903736
43903737
43903738
43903739
43903740
43903741
43903742
43903743
43903744
43903745
43903746
43903747
43903748
43903749
43903750
43903751
43903752
43903753
43903754
43903755
4390

In [ ]:
feeling_counts_df = pd.DataFrame.from_dict(feeling_counts, orient='index')


In [ ]:
feeling_counts_df.to_excel('feeling_counts_raw.xlsx')


In [ ]:
feeling_counts_df.head()

,Skepticism,Serenity,Fear,Disgust,Solidarity,Elation,Envy,Puzzlement,Recklessness,Loyalty,...,Curiosity,Regret,Nostalgia,Irony,Despair,Resignation,Closeness,Adrenaline,Belonging,Anger
tt0042192,5,0,29,10,20,29,53,2,3,11,...,17,9,3,23,1,2,3,0,0,113
tt1022603,4,1,16,5,86,252,51,7,4,3,...,23,36,6,13,11,0,8,0,2,91
tt0209144,42,1,66,13,305,100,89,221,1,15,...,176,123,3,42,16,1,13,5,6,455
tt0377092,4,0,20,13,54,91,35,2,0,9,...,7,28,7,10,3,0,5,0,2,147
tt0078788,11,7,106,31,85,49,63,16,7,14,...,65,84,6,35,27,7,15,0,8,515


In [ ]:
feeling_counts_raw_df = feeling_counts_df.copy()

In [ ]:
tconst_counts = all_r.groupby('tconst').size().reset_index(name='tconst_count')
feeling_counts_df = pd.merge(feeling_counts_df, tconst_counts, left_index=True, right_on='tconst', how='left')

In [ ]:
feeling_counts_df.head()

,Skepticism,Serenity,Fear,Disgust,Solidarity,Elation,Envy,Puzzlement,Recklessness,Loyalty,...,Nostalgia,Irony,Despair,Resignation,Closeness,Adrenaline,Belonging,Anger,tconst,tconst_count
103,5,0,29,10,20,29,53,2,3,11,...,3,23,1,2,3,0,0,113,tt0042192,470
1266,4,1,16,5,86,252,51,7,4,3,...,6,13,11,0,8,0,2,91,tt1022603,667
1040,42,1,66,13,305,100,89,221,1,15,...,3,42,16,1,13,5,6,455,tt0209144,2411
1143,4,0,20,13,54,91,35,2,0,9,...,7,10,3,0,5,0,2,147,tt0377092,758
492,11,7,106,31,85,49,63,16,7,14,...,6,35,27,7,15,0,8,515,tt0078788,1274


In [ ]:
feeling_counts_df['tconst'].nunique()

1570

In [ ]:
feeling_counts_df[feeling_counts_df['tconst_count'] < 2].shape

(1, 52)

In [ ]:
len(feeling_counts_df.columns[:-2])

50

In [ ]:

feeling_columns = feeling_counts_df.columns[:-2]
feeling_counts_df[feeling_columns] = feeling_counts_df[feeling_columns].astype('float64')

for index, row in feeling_counts_df.iterrows():
    tconst_count = row['tconst_count']
    if tconst_count > 0:
        feeling_counts_df.loc[index, feeling_columns] = (row[feeling_columns] / tconst_count).astype('float64')

In [ ]:
feeling_counts_df.head()

,Skepticism,Serenity,Fear,Disgust,Solidarity,Elation,Envy,Puzzlement,Recklessness,Loyalty,...,Nostalgia,Irony,Despair,Resignation,Closeness,Adrenaline,Belonging,Anger,tconst,tconst_count
103,0.010638,0.000000,0.061702,0.021277,0.042553,0.061702,0.112766,0.004255,0.006383,0.023404,...,0.006383,0.048936,0.002128,0.004255,0.006383,0.000000,0.000000,0.240426,tt0042192,470
1266,0.005997,0.001499,0.023988,0.007496,0.128936,0.377811,0.076462,0.010495,0.005997,0.004498,...,0.008996,0.019490,0.016492,0.000000,0.011994,0.000000,0.002999,0.136432,tt1022603,667
1040,0.017420,0.000415,0.027375,0.005392,0.126504,0.041477,0.036914,0.091663,0.000415,0.006221,...,0.001244,0.017420,0.006636,0.000415,0.005392,0.002074,0.002489,0.188718,tt0209144,2411
1143,0.005277,0.000000,0.026385,0.017150,0.071240,0.120053,0.046174,0.002639,0.000000,0.011873,...,0.009235,0.013193,0.003958,0.000000,0.006596,0.000000,0.002639,0.193931,tt0377092,758
492,0.008634,0.005495,0.083203,0.024333,0.066719,0.038462,0.049451,0.012559,0.005495,0.010989,...,0.004710,0.027473,0.021193,0.005495,0.011774,0.000000,0.006279,0.404239,tt0078788,1274


In [ ]:

feeling_counts_df.to_excel('feeling_counts_normalized.xlsx')


In [ ]:
import numpy as np

In [ ]:
df = pd.DataFrame(emotion_word_groups)

df.to_excel("emotion_word_groups.xlsx", index=False)

In [ ]:
ewg = emotion_word_groups



```
# Finding feelings through transformers

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
from datasets import Dataset


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

bart_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1
)

nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-distilroberta-base").to(device)
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-distilroberta-base")

Using device: cuda


Device set to use cuda:0


In [ ]:
omdb = pd.read_excel("bart_omdb_with_rt.xlsx")
movie_plots = omdb["typlot"].tolist()

KeyError: 'typlot'

In [ ]:
emotions = ['skepticism', 'serenity', 'fear', 'disgust', 'solidarity', 'elation', 'envy', 'puzzlement', 'recklessness', 'loyalty', 'trust', 'sadness', 'shame', 'catharsis', 'unease', 'introspection', 'excitement', 'riskiness', 'mischief', 'enlightenment', 'love', 'tension', 'sarcasm', 'frustration', 'longing', 'awe', 'defiance', 'amusement', 'vulnerability', 'surprise', 'compassion', 'resentment', 'triumph', 'happiness', 'acceptance', 'bravery', 'hope', 'conflict', 'exhilaration', 'inspiration', 'curiosity', 'regret', 'nostalgia', 'irony', 'despair', 'resignation', 'closeness', 'adrenaline', 'belonging', 'anger']

In [ ]:
dataset = Dataset.from_dict({"text": movie_plots})

def shortlist_emotions(batch, i):
    print(i)
    results = bart_classifier(batch["text"], emotions, multi_label=True)
    batch["top_10_emotions"] = [
        [label for label, score in sorted(zip(res["labels"], res["scores"]), key=lambda x: x[1], reverse=True)[:10]]
        for res in results
    ]
    return batch

dataset = dataset.map(lambda batch, i: shortlist_emotions(batch, i), batched=True, batch_size=10, with_indices=True)

all_top_emotions = dataset["top_10_emotions"]

In [ ]:
all_top_emotions

[['riskiness',
  'inspiration',
  'introspection',
  'conflict',
  'belonging',
  'tension',
  'vulnerability',
  'bravery',
  'loyalty',
  'catharsis'],
 ['amusement',
  'riskiness',
  'irony',
  'catharsis',
  'recklessness',
  'vulnerability',
  'inspiration',
  'regret',
  'defiance',
  'conflict'],
 ['riskiness',
  'conflict',
  'tension',
  'inspiration',
  'bravery',
  'vulnerability',
  'loyalty',
  'recklessness',
  'closeness',
  'irony'],
 ['conflict',
  'recklessness',
  'riskiness',
  'mischief',
  'inspiration',
  'vulnerability',
  'defiance',
  'irony',
  'surprise',
  'belonging'],
 ['conflict',
  'riskiness',
  'vulnerability',
  'belonging',
  'longing',
  'inspiration',
  'irony',
  'closeness',
  'recklessness',
  'tension'],
 ['conflict',
  'riskiness',
  'defiance',
  'resentment',
  'inspiration',
  'belonging',
  'frustration',
  'recklessness',
  'anger',
  'unease'],
 ['riskiness',
  'inspiration',
  'belonging',
  'bravery',
  'triumph',
  'recklessness',
  

In [ ]:
all_top_emotions_df = pd.DataFrame(all_top_emotions)
all_top_emotions_df.to_excel("all_top_emotions.xlsx")

In [ ]:
def batch_rank_emotions(model, tokenizer, plots, emotions, batch_size=10):
    """Processes movie plots in batches for ranking emotions."""
    scores = []
    for i in range(0, len(plots), batch_size):
        batch_plots = plots[i:i + batch_size]
        batch_emotions = emotions[i:i + batch_size]

        inputs = tokenizer(batch_plots * len(batch_emotions), batch_emotions * len(batch_plots), return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            entailment_scores = logits[:, 2].tolist()

        scores.extend(zip(batch_emotions, entailment_scores))
        print(i)
    return sorted(scores, key=lambda x: x[1], reverse=True)

In [ ]:
top_emotions_per_movie = []

count = 0
for plot, emotions in zip(movie_plots, all_top_emotions):
    ranked_emotions = batch_rank_emotions(nli_model, nli_tokenizer, [plot], emotions)
    top_5 = [emotion for emotion, _ in ranked_emotions[:5]]
    top_emotions_per_movie.append(top_5)


0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [ ]:
top_emotions_per_movie_cleaned = []
for emotions in top_emotions_per_movie:
  cleaned_emotions = [emotion for emotion in emotions if emotion not in ["riskiness", "inspiration"]]
  top_emotions_per_movie_cleaned.append(cleaned_emotions[:3])


In [ ]:
top_emotions_with_plots_c_df = pd.DataFrame({
    "typlot": movie_plots,
    "top_emotions": top_emotions_per_movie_cleaned
})

In [ ]:
top_emotions_with_plots_df = pd.DataFrame({
    "typlot": movie_plots,
    "top_emotions": top_emotions_per_movie
})

In [ ]:
top_emotions_with_plots_c_df.to_excel("clean_top_emotions_with_plots.xlsx")

# GPT feelings to clean

In [ ]:
gptf_raw = pd.read_excel("gpt_f_to_clean.xlsx")

In [ ]:
gptf_raw.head()

,title,f1,f2,f3,f4,f5,f6
0,The Cabinet of Dr. Caligari (1920),Unease,Puzzlement,Fear,NaN,NaN,NaN
1,The Kid (1921),Love,Sadness,Happiness,NaN,NaN,NaN
2,Nosferatu: A Symphony of Horror (1922),Fear,Unease,Tension,NaN,NaN,NaN
3,Safety Last! (1923),Excitement,Exhilaration,Amusement,NaN,NaN,NaN
4,Sherlock Jr. (1924),Mischief,Amusement,Surprise,NaN,NaN,NaN


In [ ]:
for col in ['f1', 'f2', 'f3', 'f4', 'f5', 'f6']:
    if col in gptf_raw.columns:
        gptf_raw[col] = gptf_raw[col].str.lower()

In [ ]:
import numpy as np
for col in ['f1', 'f2', 'f3', 'f4', 'f5', 'f6']:
    if col in gptf_raw.columns:
        gptf_raw[col] = gptf_raw[col].apply(lambda x: x if x in emotions else np.nan)


In [ ]:

gptf_dict = {}
for index, row in gptf_raw.iterrows():
    title = row['title']
    values = row[['f1', 'f2', 'f3', 'f4', 'f5', 'f6']].dropna().tolist()
    gptf_dict[title] = list(set(values))


In [ ]:
for title, emotions in gptf_dict.items():
    if len(emotions) > 3:
        gptf_dict[title] = emotions[:3]

In [ ]:
import pandas as pd

gptf_df = pd.DataFrame.from_dict(gptf_dict, orient='index')

In [ ]:
gptf_dict

{'The Cabinet of Dr. Caligari (1920)': ['puzzlement', 'unease', 'fear'],
 'The Kid (1921)': ['sadness', 'love', 'happiness'],
 'Nosferatu: A Symphony of Horror (1922)': ['unease', 'tension', 'fear'],
 'Safety Last! (1923)': ['amusement', 'exhilaration', 'excitement'],
 'Sherlock Jr. (1924)': ['amusement', 'surprise', 'mischief'],
 'Battleship Potemkin (1925)': ['defiance', 'solidarity', 'tension'],
 'The Gold Rush (1925)': ['amusement', 'hope', 'longing'],
 'The Lost World (1925)': ['awe', 'curiosity', 'excitement'],
 'Metropolis (1927)': ['awe', 'conflict', 'enlightenment'],
 'The General (1926)': ['amusement', 'bravery', 'excitement'],
 'The Jazz Singer (1927)': ['longing', 'nostalgia', 'acceptance'],
 'Sunrise (1927)': ['love', 'regret', 'catharsis'],
 'The Passion of Joan of Arc (1928)': ['bravery', 'catharsis', 'despair'],
 'All Quiet on the Western Front (1930)': ['solidarity',
  'resignation',
  'despair'],
 'City Lights (1931)': ['longing', 'love', 'happiness'],
 'Dracula (1931

In [ ]:
gptf_df.to_excel("gptf_df.xlsx")

# Creating the maximal dataset

In [ ]:
merged_df.head()

,title,title_year,year,genre,theme,theme2,theme3,t_id,imdb_id,original_language,genres,production_countries,keywords
0,City Lights,City Lights (1931),1931,Drama,Love Story,Rom-Com,NaN,901,tt0021749,en,"Comedy, Romance, Drama",United States of America,"blindness and impaired vision, eye operation, ..."
1,As Good as It Gets,As Good as It Gets (1997),1997,Drama,Love Story,Rom-Com,NaN,2898,tt0119822,en,"Drama, Comedy, Romance",United States of America,"friendship, waitress, single parent, restauran..."
2,Amélie,Amélie (2001),2001,Drama,Love Story,Rom-Com,NaN,194,tt0211915,fr,"Comedy, Romance","France, Germany","daughter, paris, france, love triangle, photog..."
3,Pretty Woman,Pretty Woman (1990),1990,Drama,Love Story,Rom-Com,NaN,114,tt0100405,en,"Comedy, Romance",United States of America,"hotel, friendship, prostitute, sports car, cap..."
4,Manhattan,Manhattan (1979),1979,Drama,Love Story,Rom-Com,NaN,696,tt0079522,en,"Comedy, Drama, Romance",United States of America,"adultery, lolita, new york city, based on nove..."


In [ ]:
pti_df_short.head()

,imdb_id,title,year,title_year,original_language,production_countries,imdb_rating
0,tt0021749,City Lights,1931,City Lights (1931),en,United States of America,8.5
1,tt0119822,As Good as It Gets,1997,As Good as It Gets (1997),en,United States of America,7.7
2,tt0211915,Amélie,2001,Amélie (2001),fr,"France, Germany",8.3
3,tt0100405,Pretty Woman,1990,Pretty Woman (1990),en,United States of America,7.1
4,tt0079522,Manhattan,1979,Manhattan (1979),en,United States of America,7.8


In [ ]:
omdb.columns

Index(['Unnamed: 0', 'Title', 'Year', 'Rated', 'Released', 'Runtime', 'og',
       'Director', 'Writer', 'Actors', 'Plot', 'Language', 'Country', 'Awards',
       'Poster', 'Ratings', 'Metascore', 'imdbRating', 'RT', 'imdbVotes',
       'imdbID', 'Type', 'DVD', 'BoxOffice', 'Production', 'Website',
       'Response'],
      dtype='object')

In [ ]:
omdb = omdb.rename(columns={'imdbID': 'imdb_id'})

In [ ]:
omdb_short = omdb[['imdb_id', 'Plot', 'Metascore', 'RT']]

In [ ]:
ptio_df = pd.merge(pti_df_short, omdb_short, on='imdb_id', how='left')

In [ ]:
ptio_df.columns

Index(['imdb_id', 'title', 'year', 'title_year', 'original_language',
       'production_countries', 'imdb_rating', 'Plot', 'Metascore', 'RT'],
      dtype='object')

In [ ]:
ptio_df = ptio_df[['imdb_id', 'title', 'year', 'title_year', 'original_language', 'production_countries', 'Plot', 'imdb_rating', 'RT', 'Metascore']]

In [ ]:
ptio_df = ptio_df.rename(columns={'original_language':'language', 'Plot':'plot', "Metascore":'metascore', 'RT':'tomatometer', 'production_countries': 'countries'})

In [ ]:
ptio_df.columns

Index(['imdb_id', 'title', 'year', 'title_year', 'language', 'countries',
       'plot', 'imdb_rating', 'tomatometer', 'metascore'],
      dtype='object')

In [ ]:
ptio_df.head()

,imdb_id,title,year,title_year,language,countries,plot,imdb_rating,tomatometer,metascore
0,tt0021749,City Lights,1931,City Lights (1931),en,United States of America,A tramp falls in love with a beautiful blind g...,8.5,0.95,99.0
1,tt0119822,As Good as It Gets,1997,As Good as It Gets (1997),en,United States of America,"New York City. Melvin Udall, a cranky, bigoted...",7.7,0.86,67.0
2,tt0211915,Amélie,2001,Amélie (2001),fr,"France, Germany",Amélie is a story about a girl named Amélie wh...,8.3,0.90,70.0
3,tt0100405,Pretty Woman,1990,Pretty Woman (1990),en,United States of America,Because of his extreme wealth and suave good l...,7.1,0.64,51.0
4,tt0079522,Manhattan,1979,Manhattan (1979),en,United States of America,Forty-two year old Isaac Davis has a romantici...,7.8,0.93,83.0


In [ ]:
ptio_df['imdb_rating'] = ptio_df['imdb_rating'] * 10
ptio_df['tomatometer'] = ptio_df['tomatometer'] * 100


In [ ]:
ptio_df['expert_avg_rating'] = (ptio_df['tomatometer'] + ptio_df['metascore']) / 2
ptio_df.loc[ptio_df['tomatometer'].isna(), 'expert_avg_rating'] = ptio_df['metascore']
ptio_df.loc[ptio_df['metascore'].isna(), 'expert_avg_rating'] = ptio_df['tomatometer']

ptio_df['avg_rating'] = (ptio_df['expert_avg_rating'] + ptio_df['imdb_rating']) / 2
ptio_df.loc[ptio_df['imdb_rating'].isna(), 'avg_rating'] = ptio_df['expert_avg_rating']
ptio_df.loc[ptio_df['expert_avg_rating'].isna(), 'avg_rating'] = ptio_df['imdb_rating']


In [ ]:
ptio_df.to_excel("max_ds_without_preds.xlsx")

Now Link genres

In [ ]:
genres = pd.read_excel("genres.xlsx")

In [ ]:
genres = genres[['imdb_id', 'combined']]

In [ ]:
genres = genres.rename(columns={'combined': 'genres'})

In [ ]:
ptio_w_genres = pd.merge(ptio_df, genres, on='imdb_id', how='left')

In [ ]:
ptio_w_genres['genres'].isna().sum()

0

In [ ]:
ptio_w_genres['genres'][200:230]

,genres
200,"['history', 'action', 'biography', 'war', 'dra..."
201,"['comedy', 'romance', 'drama']"
202,"['history', 'romance', 'biography', 'drama']"
203,"['history', 'romance', 'biography', 'drama']"
204,"['western', 'drama']"
205,"['war', 'adventure', 'romance', 'drama']"
206,"['history', 'biography', 'drama']"
207,"['history', 'drama']"
208,"['history', 'biography', 'drama']"
209,"['history', 'drama']"


In [ ]:
unique_genres = set()
for index, row in ptio_w_genres.iterrows():
    genres_list_str = row['genres']
    genres_list_str = genres_list_str.strip('[')
    genres_list_str = genres_list_str.strip(']')
    genres_list = [genre.strip("'") for genre in genres_list_str.split(', ')]

    genres_list_str = genres_list_str.strip("'")
    unique_genres.update(genres_list)

unique_genres

{'action',
 'adventure',
 'animation',
 'biography',
 'comedy',
 'crime',
 'documentary',
 'drama',
 'family',
 'fantasy',
 'film-noir',
 'history',
 'horror',
 'music',
 'musical',
 'mystery',
 'romance',
 'sci-fi',
 'sport',
 'thriller',
 'true_story',
 'war',
 'western'}

In [ ]:
len(unique_genres)

23

In [ ]:

for genre in unique_genres:
    ptio_w_genres[genre] = 0

for index, row in ptio_w_genres.iterrows():
    genres_list_str = row['genres']
    if isinstance(genres_list_str, str):
      genres_list_str = genres_list_str.strip('[')
      genres_list_str = genres_list_str.strip(']')
      genres_list = [genre.strip("'") for genre in genres_list_str.split(', ')]
      for genre in genres_list:
          if genre in unique_genres:
              ptio_w_genres.loc[index, genre] = 1


In [ ]:
ptio_w_genres.to_excel("max_ds_with_genres.xlsx")

NameError: name 'ptio_w_genres' is not defined

Add TMDB themes (1915 themes that appear in at least 3 movies)

In [ ]:
ptio_w_genres = pd.read_excel("max_ds_with_genres.xlsx")

In [ ]:
ptio_w_genres.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
ptio_w_genres.columns

Index(['imdb_id', 'title', 'year', 'title_year', 'language', 'countries',
       'plot', 'imdb_rating', 'tomatometer', 'metascore', 'expert_avg_rating',
       'avg_rating', 'genres', 'drama', 'sci-fi', 'musical', 'romance',
       'western', 'animation', 'sport', 'comedy', 'crime', 'horror', 'fantasy',
       'documentary', 'biography', 'adventure', 'history', 'film-noir',
       'music', 'war', 'true_story', 'action', 'mystery', 'thriller',
       'family'],
      dtype='object')

In [ ]:
merged_df.columns

Index(['title', 'title_year', 'year', 'pc_genre', 'theme', 'theme2', 'theme3',
       'tmdb_id', 'imdb_id', 'original_language', 'tmdb_genres',
       'production_countries', 'tmdb_themes'],
      dtype='object')

In [ ]:
th

In [ ]:
t1_themes_df = merged_df[['imdb_id', 'tmdb_themes']]

In [ ]:
ptio_w_genres_t1 = pd.merge(ptio_w_genres, t1_themes_df, on='imdb_id', how='left')

In [ ]:
ptio_w_genres_t1.columns

Index(['imdb_id', 'title', 'year', 'title_year', 'language', 'countries',
       'plot', 'imdb_rating', 'tomatometer', 'metascore', 'expert_avg_rating',
       'avg_rating', 'genres', 'drama', 'sci-fi', 'musical', 'romance',
       'western', 'animation', 'sport', 'comedy', 'crime', 'horror', 'fantasy',
       'documentary', 'biography', 'adventure', 'history', 'film-noir',
       'music', 'war', 'true_story', 'action', 'mystery', 'thriller', 'family',
       'tmdb_themes'],
      dtype='object')

In [ ]:
ptio_w_genres_t1['tmdb_themes']

,tmdb_themes
0,"blindness and impaired vision, eye operation, ..."
1,"friendship, waitress, single parent, restauran..."
2,"daughter, paris, france, love triangle, photog..."
3,"hotel, friendship, prostitute, sports car, cap..."
4,"adultery, lolita, new york city, based on nove..."
...,...
1598,"experiment, detective, cave, missouri, investi..."
1599,"kidnapping, externally controlled action, mani..."
1600,"daughter, funeral, loss of loved one, ritual, ..."
1601,"chess, prisoner of war, hungary, black and whi..."


In [ ]:
ptio_w_genres_t1['tmdb_themes'] = ptio_w_genres_t1['tmdb_themes'].str.replace(', ', ';;')

In [ ]:
unique_t1_themes = set()
for index, row in ptio_w_genres_t1.iterrows():
    if isinstance(row['tmdb_themes'], str):
        t1_themes_list_str = row['tmdb_themes']
        t1_themes_list = [genre for genre in t1_themes_list_str.split(';;')] #split by ',' and strip single quotes
        unique_t1_themes.update(t1_themes_list)

unique_t1_themes

{'nosferatu',
 'water skiing',
 'pest control',
 'emperor',
 'bodily dismemberment',
 'arabian',
 'bible quote',
 'based on magazine',
 'holiday season',
 'generation z',
 'love at first sight',
 'flea',
 'religious art',
 'courtroom drama',
 'gentrification',
 'cheyenne',
 'thirty something',
 'graduation',
 'giving away money',
 'wild west',
 'swinging 60s',
 'east india company',
 'mutant',
 'montana',
 'shopping',
 '5th century',
 'inventor',
 'drug humor',
 'cheerleader',
 'charming man',
 'botched robbery',
 'quest for identity',
 'pep rally',
 'pulitzer prize',
 'human sacrifice',
 'geologist',
 'sex party',
 'stapler',
 'chemist',
 'gang violence',
 'teenage mortality',
 '12th century',
 'cape cod',
 'testosterone',
 'approved school ',
 'fishing boat',
 'napoleon bonaparte',
 'water slide',
 'stockholm syndrome',
 'vaudeville',
 'dreary',
 'boy and dog',
 'existentialism',
 'terror cell',
 'taj mahal',
 'cattle',
 'heavy metal',
 'sword',
 'externally controlled action',
 'oil

In [ ]:
len(unique_t1_themes)

7570

In [ ]:
ptio_w_genres_all_t1 = ptio_w_genres_t1.copy()

In [ ]:
len(keywords_atleast)

1915

In [ ]:
theme_data = {}
for index, row in ptio_w_genres_all_t1.iterrows():
    theme_t1s_list_str = row['tmdb_themes']
    if isinstance(theme_t1s_list_str, str):
        theme_t1s_list = [theme_t1 for theme_t1 in theme_t1s_list_str.split(';;')]


        theme_data[index] = {theme: 1 for theme in theme_t1s_list if theme in keywords_atleast}
        theme_data[index].update({theme:0 for theme in keywords_atleast if theme not in theme_t1s_list})
    else:

        theme_data[index] = {theme: 0 for theme in keywords_atleast}

theme_df = pd.DataFrame.from_dict(theme_data, orient='index')
theme_df = theme_df.reindex(columns = list(keywords_atleast))

theme_df = theme_df.rename(
    columns={col: "t1_" + col for col in theme_df.columns if col in keywords_atleast}
)

ptio_w_genres_all_t1 = pd.concat([ptio_w_genres_all_t1, theme_df], axis=1)




In [ ]:
pwgt1 = ptio_w_genres_all_t1.copy()

In [ ]:
pwgt1.columns

Index(['imdb_id', 'title', 'year', 'title_year', 'language', 'countries',
       'plot', 'imdb_rating', 'tomatometer', 'metascore',
       ...
       't1_satanism', 't1_demonic possession', 't1_satanic cult', 't1_séance',
       't1_evil spirit', 't1_exorcism', 't1_praying', 't1_religious horror',
       't1_grim', 't1_psychic power'],
      dtype='object', length=1952)

In [ ]:
pwgt1['tmdb_themes'][65]

"secret love;;wyoming;;usa;;countryside;;homophobia;;marriage crisis;;intolerance;;in the closet;;summer;;cowboy;;lgbt;;star crossed lovers;;1960s;;closeted homosexual;;gay romance;;gay theme;;gay relationship;;boys' love (bl)"

In [ ]:
pwgt1["t1_wyoming"][64]

0

In [ ]:
pwgt1.to_excel("pwgt1.xlsx")

Exception ignored in: <function ZipFile.__del__ at 0x7a11e1086160>
Traceback (most recent call last):
  File "/usr/lib/python3.11/zipfile.py", line 1895, in __del__
    self.close()
  File "/usr/lib/python3.11/zipfile.py", line 1912, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


Add MSPT themes (70). Ensure you split by a t2_ prefix

In [ ]:
mspt = pd.read_excel("mspt_tags.xlsx")

In [ ]:
mspt.head()

,imdb_id,tags
0,tt0010323,"avant garde,gothic,murder,psychological,violen..."
1,tt0012349,NaN
2,tt0013442,"gothic,paranormal,cult,atmospheric,good versus..."
3,tt0014429,prank
4,tt0015324,"absurd,psychedelic,humor,satire"


In [ ]:
mspt['tags'] = mspt['tags'].str.replace(', ', ',')

In [ ]:
mspt['tags'].str.count(', ').sum()

0.0

In [ ]:
mspt = mspt.rename(columns={'imdbID': 'imdb_id'})

In [ ]:
pwgt12 = pd.merge(pwgt1, mspt, on='imdb_id', how='left')

In [ ]:
pwgt12 = pwgt12.rename(columns={'tags': 'mspt_themes'})

In [ ]:
unique_t2_themes = set()
for index, row in mspt.iterrows():
    if isinstance(row['tags'], str):
        t2s_str = row['tags']
        t2s_list = [genre for genre in t2s_str.split(',')]
        unique_t2_themes.update(t2s_list)

unique_t2_themes

{'absurd',
 'action',
 'adult comedy',
 'allegory',
 'alternate history',
 'alternate reality',
 'anti war',
 'atmospheric',
 'autobiographical',
 'avant garde',
 'blaxploitation',
 'bleak',
 'boring',
 'brainwashing',
 'claustrophobic',
 'clever',
 'comedy',
 'comic',
 'cruelty',
 'cult',
 'cute',
 'dark',
 'depressing',
 'dramatic',
 'entertaining',
 'fantasy',
 'feel-good',
 'flashback',
 'good versus evil',
 'gothic',
 'grindhouse film',
 'haunting',
 'historical',
 'historical fiction',
 'home movie',
 'horror',
 'humor',
 'insanity',
 'inspiring',
 'intrigue',
 'magical realism',
 'melodrama',
 'murder',
 'mystery',
 'neo noir',
 'paranormal',
 'philosophical',
 'plot twist',
 'pornographic',
 'prank',
 'psychedelic',
 'psychological',
 'queer',
 'realism',
 'revenge',
 'romantic',
 'sadist',
 'satire',
 'sci-fi',
 'sentimental',
 'storytelling',
 'stupid',
 'suicidal',
 'suspenseful',
 'thought-provoking',
 'tragedy',
 'violence',
 'western',
 'whimsical'}

In [ ]:
len(unique_t2_themes)

69

In [ ]:
theme_data = {}
for index, row in pwgt12.iterrows():
    t2s_str = row['mspt_themes']
    if isinstance(t2s_str, str):
        t2s = [theme_t2 for theme_t2 in t2s_str.split(',')]


        theme_data[index] = {theme: 1 for theme in t2s if theme in unique_t2_themes}
        theme_data[index].update({theme: 0 for theme in unique_t2_themes if theme not in t2s})
    else:

        theme_data[index] = {theme: 0 for theme in unique_t2_themes}

theme_df2 = pd.DataFrame.from_dict(theme_data, orient='index')
theme_df2 = theme_df2.reindex(columns = list(unique_t2_themes))

theme_df2 = theme_df2.rename(
    columns={col: "t2_" + col for col in theme_df2.columns if col in unique_t2_themes}
)

pwgt12 = pd.concat([pwgt12, theme_df2], axis=1)





In [ ]:
pwgt12['mspt_themes'][444]

'dramatic,inspiring,feel-good'

In [ ]:
pwgt12['t2_feel-good'][442]

0

Add review-based feelings

In [ ]:
pwgt12.to_excel("pwgt12.xlsx")

Got feeling_counts_df (normalized) from above

In [ ]:
feeling_counts_df = pd.read_excel("feeling_counts_normalized.xlsx")

In [ ]:
feeling_counts_df = feeling_counts_df.drop(columns=['Unnamed: 0'])

In [ ]:
feeling_counts_df.head()

,Skepticism,Serenity,Fear,Disgust,Solidarity,Elation,Envy,Puzzlement,Recklessness,Loyalty,...,Nostalgia,Irony,Despair,Resignation,Closeness,Adrenaline,Belonging,Anger,tconst,tconst_count
0,0.010638,0.000000,0.061702,0.021277,0.042553,0.061702,0.112766,0.004255,0.006383,0.023404,...,0.006383,0.048936,0.002128,0.004255,0.006383,0.000000,0.000000,0.240426,tt0042192,470
1,0.005997,0.001499,0.023988,0.007496,0.128936,0.377811,0.076462,0.010495,0.005997,0.004498,...,0.008996,0.019490,0.016492,0.000000,0.011994,0.000000,0.002999,0.136432,tt1022603,667
2,0.017420,0.000415,0.027375,0.005392,0.126504,0.041477,0.036914,0.091663,0.000415,0.006221,...,0.001244,0.017420,0.006636,0.000415,0.005392,0.002074,0.002489,0.188718,tt0209144,2411
3,0.005277,0.000000,0.026385,0.017150,0.071240,0.120053,0.046174,0.002639,0.000000,0.011873,...,0.009235,0.013193,0.003958,0.000000,0.006596,0.000000,0.002639,0.193931,tt0377092,758
4,0.008634,0.005495,0.083203,0.024333,0.066719,0.038462,0.049451,0.012559,0.005495,0.010989,...,0.004710,0.027473,0.021193,0.005495,0.011774,0.000000,0.006279,0.404239,tt0078788,1274


In [ ]:
new_columns = {col: 'f1_' + col.lower() for col in feeling_counts_df.columns[:-2]}
new_columns['tconst'] = 'tconst'
new_columns['tconst_count'] = 'tconst_count'
feeling_counts_df = feeling_counts_df.rename(columns=new_columns)

In [ ]:
feeling_counts_df.drop(columns=['tconst_count'], inplace=True)

In [ ]:
feeling_counts_df.rename(columns={'tconst': 'imdb_id'}, inplace=True)

In [ ]:
pwgt12 = pd.merge(pwgt12, feeling_counts_df, on='imdb_id', how='left')

In [ ]:
pwgt12f1 = pwgt12.copy()

In [ ]:
pwgt12f1.to_excel("pwgt12f1.xlsx")

Add plot-based feelings

In [ ]:
omdb.columns

Index(['Unnamed: 0', 'Title', 'Year', 'Rated', 'Released', 'Runtime', 'Genre',
       'Director', 'Writer', 'Actors', 'Plot', 'Language', 'Country', 'Awards',
       'Poster', 'Ratings', 'Metascore', 'imdbRating', 'RT', 'imdbVotes',
       'imdbID', 'Type', 'DVD', 'BoxOffice', 'Production', 'Website',
       'Response', 'ty', 'typlot'],
      dtype='object')

In [ ]:
omdb_short = omdb[['imdbID', 'typlot', 'ty']]

In [ ]:
p_emos = pd.read_excel("clean_top_emotions_with_plots.xlsx")

In [ ]:
p_emos.head()

,typlot,top_emotions
0,"The Cabinet of Dr. Caligari (1920): Francis, a...","['bravery', 'catharsis', 'vulnerability']"
1,"The Kid (1921): The opening title reads: ""A co...","['irony', 'catharsis', 'amusement']"
2,Nosferatu: A Symphony of Horror (1922): Wisbou...,"['bravery', 'irony', 'tension']"
3,"Safety Last! (1923): In 1922, the country boy ...","['mischief', 'irony', 'conflict']"
4,Sherlock Jr. (1924): A meek and mild projectio...,"['irony', 'longing', 'closeness']"


In [ ]:
p_emos = pd.merge(p_emos, omdb_short, on='typlot', how='left')

In [ ]:
p_emos.head()

,typlot,top_emotions,imdbID,ty
0,"The Cabinet of Dr. Caligari (1920): Francis, a...","['bravery', 'catharsis', 'vulnerability']",tt0010323,The Cabinet of Dr. Caligari (1920)
1,"The Kid (1921): The opening title reads: ""A co...","['irony', 'catharsis', 'amusement']",tt0012349,The Kid (1921)
2,Nosferatu: A Symphony of Horror (1922): Wisbou...,"['bravery', 'irony', 'tension']",tt0013442,Nosferatu: A Symphony of Horror (1922)
3,"Safety Last! (1923): In 1922, the country boy ...","['mischief', 'irony', 'conflict']",tt0014429,Safety Last! (1923)
4,Sherlock Jr. (1924): A meek and mild projectio...,"['irony', 'longing', 'closeness']",tt0015324,Sherlock Jr. (1924)


In [ ]:
p_emos['imdbID'].isna().sum()

0

In [ ]:
counts = p_emos['imdbID'].value_counts()
counts[counts > 1].sum()

0

In [ ]:
p_emos.rename(columns={'imdbID': 'imdb_id'}, inplace=True)

In [ ]:
p_emos_short = p_emos[['imdb_id', 'top_emotions']]

In [ ]:
p_emos_short.to_excel("p_emos_short.xlsx")

In [ ]:
import pandas as pd

In [ ]:
p_emos_short = pd.read_excel("p_emos_short.xlsx")

In [ ]:
pwgt12f1 = pd.read_excel("pwgt12f1.xlsx")

In [ ]:
p_emos_short.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
p_emos_short.head()

,Unnamed: 0,imdb_id,top_emotions
0,0,tt0010323,"['bravery', 'catharsis', 'vulnerability']"
1,1,tt0012349,"['irony', 'catharsis', 'amusement']"
2,2,tt0013442,"['bravery', 'irony', 'tension']"
3,3,tt0014429,"['mischief', 'irony', 'conflict']"
4,4,tt0015324,"['irony', 'longing', 'closeness']"


In [ ]:
unique_emos = set()
for index, row in p_emos_short.iterrows():
    emo_list_str = row['top_emotions']
    emo_list_str = emo_list_str.strip('[')
    emo_list_str = emo_list_str.strip(']')
    emo_list = [genre.strip("'") for genre in emo_list_str.split(', ')]

    unique_emos.update(emo_list)

unique_emos

{'acceptance',
 'amusement',
 'anger',
 'awe',
 'belonging',
 'bravery',
 'catharsis',
 'closeness',
 'compassion',
 'conflict',
 'curiosity',
 'defiance',
 'despair',
 'disgust',
 'elation',
 'envy',
 'excitement',
 'exhilaration',
 'fear',
 'frustration',
 'happiness',
 'hope',
 'introspection',
 'irony',
 'longing',
 'love',
 'loyalty',
 'mischief',
 'nostalgia',
 'puzzlement',
 'recklessness',
 'regret',
 'resentment',
 'sadness',
 'sarcasm',
 'serenity',
 'shame',
 'skepticism',
 'solidarity',
 'surprise',
 'tension',
 'triumph',
 'trust',
 'unease',
 'vulnerability'}

In [ ]:
len(unique_emos)

45

In [ ]:
all_emos = []

for k, v in ewr.items():
    all_emos.append(k.lower())

all_emos

['skepticism',
 'serenity',
 'fear',
 'disgust',
 'solidarity',
 'elation',
 'envy',
 'puzzlement',
 'recklessness',
 'loyalty',
 'trust',
 'sadness',
 'shame',
 'catharsis',
 'unease',
 'introspection',
 'excitement',
 'riskiness',
 'mischief',
 'enlightenment',
 'love',
 'tension',
 'sarcasm',
 'frustration',
 'longing',
 'awe',
 'defiance',
 'amusement',
 'vulnerability',
 'surprise',
 'compassion',
 'resentment',
 'triumph',
 'happiness',
 'acceptance',
 'bravery',
 'hope',
 'conflict',
 'exhilaration',
 'inspiration',
 'curiosity',
 'regret',
 'nostalgia',
 'irony',
 'despair',
 'resignation',
 'closeness',
 'adrenaline',
 'belonging',
 'anger']

In [ ]:
p_emos_short['top_emotions'] = p_emos_short['top_emotions'].astype(str).str.replace("[", "", regex=False)
p_emos_short['top_emotions'] = p_emos_short['top_emotions'].astype(str).str.replace("'", "", regex=False)
p_emos_short['top_emotions'] = p_emos_short['top_emotions'].astype(str).str.replace("]", "", regex=False)


In [ ]:
p_emos_short['top_emotions']

,top_emotions
0,"bravery, catharsis, vulnerability"
1,"irony, catharsis, amusement"
2,"bravery, irony, tension"
3,"mischief, irony, conflict"
4,"irony, longing, closeness"
...,...
1598,"sarcasm, mischief, recklessness"
1599,"awe, unease, tension"
1600,"vulnerability, surprise, tension"
1601,"surprise, solidarity, bravery"


In [ ]:
missing_f2_emos = set(all_emos) - set(unique_emos)

In [ ]:
emo_f2_data = {}
for index, row in p_emos_short.iterrows():
    emo_list_str = row['top_emotions']
    if isinstance(emo_list_str, str):
        emo_list = [emo for emo in emo_list_str.split(', ')]


        emo_f2_data[index] = {emo: 1 for emo in emo_list if emo in all_emos}
        emo_f2_data[index].update({emo: 0 for emo in all_emos if emo not in emo_list})
    else:
        emo_f2_data[index] = {emo: 0 for emo in all_emos}


emo_f2_df = pd.DataFrame.from_dict(emo_f2_data, orient='index')
emo_f2_df = emo_f2_df.reindex(columns=list(all_emos))

p_emos_short = pd.concat([p_emos_short, emo_f2_df], axis=1)

p_emos_short = p_emos_short.rename(
    columns={col: "f2_" + col for col in p_emos_short.columns if col in all_emos}
)

In [ ]:
emo_list

['bravery', 'tension', 'vulnerability']

In [ ]:
p_emos_short['top_emotions'][68]

'irony, mischief, frustration'

In [ ]:
p_emos_short['f2_irony'][67]

0

In [ ]:
p_emos_short.head()

,imdb_id,top_emotions,f2_skepticism,f2_serenity,f2_fear,f2_disgust,f2_solidarity,f2_elation,f2_envy,f2_puzzlement,...,f2_curiosity,f2_regret,f2_nostalgia,f2_irony,f2_despair,f2_resignation,f2_closeness,f2_adrenaline,f2_belonging,f2_anger
0,tt0010323,"bravery, catharsis, vulnerability",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,tt0012349,"irony, catharsis, amusement",0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,tt0013442,"bravery, irony, tension",0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,tt0014429,"mischief, irony, conflict",0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,tt0015324,"irony, longing, closeness",0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0


In [ ]:
p_emos_short.rename(columns={'top_emotions': 'plot_feelings'}, inplace=True)

In [ ]:
pwgt12f1 = pwgt12f1.drop(columns=['Unnamed: 0'])

In [ ]:
pwgtt12f12 = pd.merge(pwgt12f1, p_emos_short, on='imdb_id', how='left')

In [ ]:
pwgtt12f12.to_excel("pwgtt12f12.xlsx")

Add GPT-based feelings

In [ ]:
gptf = pd.read_excel('gptf_df_clean.xlsx')

In [ ]:
gptf.columns

Index(['ty', 'imdb_id', 'f'], dtype='object')

In [ ]:
gptf.drop(columns=['ty'], inplace=True)

In [ ]:
emo_f3_data = {}
for index, row in gptf.iterrows():
    emo_list_str = row['f']
    if isinstance(emo_list_str, str):
        emo_list = [emo for emo in emo_list_str.split(',')]


        emo_f3_data[index] = {emo: 1 for emo in emo_list if emo in all_emos}
        emo_f3_data[index].update({emo: 0 for emo in all_emos if emo not in emo_list})
    else:

        emo_f3_data[index] = {emo: 0 for emo in all_emos}


emo_f3_df = pd.DataFrame.from_dict(emo_f3_data, orient='index')
emo_f3_df = emo_f3_df.reindex(columns=list(all_emos))

gptf = pd.concat([gptf, emo_f3_df], axis=1)

gptf = gptf.rename(
    columns={col: "f3_" + col for col in gptf.columns if col in all_emos}
)

In [ ]:
gptf.columns

Index(['imdb_id', 'f', 'f3_skepticism', 'f3_serenity', 'f3_fear', 'f3_disgust',
       'f3_solidarity', 'f3_elation', 'f3_envy', 'f3_puzzlement',
       'f3_recklessness', 'f3_loyalty', 'f3_trust', 'f3_sadness', 'f3_shame',
       'f3_catharsis', 'f3_unease', 'f3_introspection', 'f3_excitement',
       'f3_riskiness', 'f3_mischief', 'f3_enlightenment', 'f3_love',
       'f3_tension', 'f3_sarcasm', 'f3_frustration', 'f3_longing', 'f3_awe',
       'f3_defiance', 'f3_amusement', 'f3_vulnerability', 'f3_surprise',
       'f3_compassion', 'f3_resentment', 'f3_triumph', 'f3_happiness',
       'f3_acceptance', 'f3_bravery', 'f3_hope', 'f3_conflict',
       'f3_exhilaration', 'f3_inspiration', 'f3_curiosity', 'f3_regret',
       'f3_nostalgia', 'f3_irony', 'f3_despair', 'f3_resignation',
       'f3_closeness', 'f3_adrenaline', 'f3_belonging', 'f3_anger'],
      dtype='object')

In [ ]:
max_ds = pd.merge(pwgtt12f12, gptf, on='imdb_id', how='left')

In [ ]:
max_ds.rename(columns={'f': 'gpt_feelings'}, inplace=True)

In [ ]:
max_ds.to_excel("max_ds.xlsx")